In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

# import data

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_01.h5ad")
adata_scraw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
adata_scraw

In [ ]:
# preprocessing pipeline
sc.pp.normalize_total(adata_scraw)
sc.pp.log1p(adata_scraw)
adata_scraw.raw = adata_scraw.copy()

sc.pp.highly_variable_genes(adata_scraw)
adata_scraw = adata_scraw[:, adata_scraw.var['highly_variable']]

sc.pp.scale(adata_scraw, max_value=10)
sc.tl.pca(adata_scraw, svd_solver='arpack')

In [ ]:
# t-sne 
sc.tl.tsne(adata_scraw, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_scraw, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_scraw)
sc.pl.umap(adata_scraw, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_scraw.obsm['X_pca'],adata_scraw.obs, 'sample')

# Harmony 결과를 저장
adata_scraw.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_scraw, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_scraw)
sc.pl.umap(adata_scraw, color=['state'])

In [ ]:
top_genes_filtered = [g for g in top_genes_state if g in adata_scraw.var_names]

sc.pl.heatmap(
    adata_scraw,
    var_names=top_genes_filtered,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    figsize=(10, 7)
)

In [ ]:
missing_genes = [gene for gene in top_genes_state if gene not in adata_scraw.var_names]
print(f"❌ Missing genes: {missing_genes}")

In [ ]:
rank_df_state

In [ ]:
rank_df_state.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/scTotal_State_DEG.csv")

In [ ]:
adata_scraw.obs

In [ ]:
adata_scraw.obs['celltypist_liver'] = adata_scrna.obs['celltypist_liver']

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")
sc.pp.normalize_total(adata_scraw, target_sum=1e4)
sc.pp.log1p(adata_scraw)
adata_scrna.raw = adata_scraw

In [ ]:
sc.tl.leiden(adata_scrna, resolution=0.5, key_added="leiden")
sc.pl.umap(adata_scrna, color=['leiden'])

In [ ]:
genes = ['TWIST1'] 
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))

if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scrna,
        keys=gene,
        groupby='state',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sc.pl.umap(adata_scrna, color = 'TWIST1')

In [ ]:
adata_scrna

In [ ]:
sc.pl.umap(adata_scrna, color = '11cluster')

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='leiden',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
predictions_cluster = celltypist.annotate(adata_scrna, model='Healthy_Human_Liver.pkl', majority_voting=True, over_clustering=adata_scrna.obs["leiden"]
)
adata_scrna =predictions_cluster.to_adata()

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='majority_voting',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
adata_scrna.obs['celltype'] = adata_scrna.obs['majority_voting'] 

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='celltypist_liver',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='PTPRC',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='CD45+',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='SLC7A5',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='SLC7A5 expression',
    size=10
)

In [ ]:
sc.pl.umap(adata_scrna[adata_scrna.obs['state'] == 'healthy'], color='celltype', title='Healthy')
sc.pl.umap(adata_scrna[adata_scrna.obs['state'] == 'NAFLD'], color='celltype', title='MASLD')

In [ ]:
sc.pl.umap(adata_scrna[adata_scrna.obs['state'] == 'healthy'], color='SLC7A5', title='Healthy')
sc.pl.umap(adata_scrna[adata_scrna.obs['state'] == 'NAFLD'], color='SLC7A5', title='MASLD')

In [ ]:
adata_scrna.obs['state'].value_counts()

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_scrna.obs['celltype']
batch_labels = adata_scrna.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_02.h5ad")

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_02.h5ad")

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")
adata_scraw.obs['celltype']=adata_scrna.obs['celltype']

In [ ]:
adata_T = adata_scrna[adata_scrna.obs['celltype'].isin(['T cells'])].copy()
adata_T

In [ ]:
adata_scrna.obs

In [ ]:
sc.tl.dendrogram(adata_scrna,groupby='state')
sc.tl.rank_genes_groups(adata_scrna, groupby='state', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scrna, n_genes=10)

rank_df_state = sc.get.rank_genes_groups_df(adata_scrna, group=None)
top_genes = rank_df_state.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scrna,
    var_names=top_genes,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df_state

In [ ]:
rank_df_state.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/scTotal_state_DEG.csv")

# scRNA _ mono/MAC analysis

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
adata_scraw.obs['celltype']=adata_scrna.obs['celltype']

In [ ]:
adata_scraw.obs['celltype'].value_counts()

In [ ]:
adata_scmono = adata_scraw[adata_scraw.obs['celltype'].isin(['Macrophages','Mono+mono derived cells'])].copy()
adata_scmono

In [ ]:
adata_scmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono01.h5ad")

In [ ]:
adata_scmono=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono01.h5ad")

In [ ]:
adata_scmono.raw = adata_scmono

In [ ]:
sc.pp.normalize_total(adata_scmono, target_sum=1e4)
sc.pp.log1p(adata_scmono)
sc.pp.highly_variable_genes(adata_scmono, flavor='seurat', n_top_genes=2000)

In [ ]:
# Scale data
sc.pp.scale(adata_scmono, max_value=10)
# Run PCA
sc.tl.pca(adata_scmono, svd_solver='arpack')
print(adata_scmono.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_scmono, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_scmono.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_scmono, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_scmono, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_scmono)
sc.pl.umap(adata_scmono, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_scmono.obsm['X_pca'],adata_scmono.obs, 'sample')

# Harmony 결과를 저장
adata_scmono.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_scmono, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_scmono)

In [ ]:
sc.pl.umap(adata_scmono,color='state')

In [ ]:
sc.tl.rank_genes_groups(
    adata_scmono,
    groupby="state",        # 비교할 obs 열
    groups=["NAFLD"],     # 비교할 그룹
    reference="healthy",     # 참조 그룹
    use_raw=True,
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
deg_df = sc.get.rank_genes_groups_df(adata_scmono, group="NAFLD")
print(deg_df.head())

In [ ]:
filtered = deg_df[
    (deg_df['logfoldchanges'] > 0.58) &
    (deg_df['pvals_adj'] < 0.05)
]

In [ ]:
top20_filtered = filtered.sort_values(by='scores', ascending=False).head(50)
print(top20_filtered)

In [ ]:
deg_df['-log10(padj)'] = -np.log10(deg_df['pvals_adj'])
genes_of_interest = ["SLC7A5"]

plt.figure(figsize=(7, 6))
plt.scatter(
    deg_df['logfoldchanges'], 
    deg_df['-log10(padj)'], 
    c="grey", s=10, alpha=0.7
)

# 유의미한 cutoff 설정 (예: logFC > 1, padj < 0.05)
sig_up = (deg_df['logfoldchanges'] > 1) & (deg_df['pvals_adj'] < 0.05)
sig_down = (deg_df['logfoldchanges'] < -1) & (deg_df['pvals_adj'] < 0.05)

plt.scatter(deg_df.loc[sig_up, 'logfoldchanges'],
            deg_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")

plt.scatter(deg_df.loc[sig_down, 'logfoldchanges'],
            deg_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

# cutoff 선
plt.axvline(1, color="black", linestyle="--")
plt.axvline(-1, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: MASLD vs Normal")
plt.legend()

for gene in genes_of_interest:
    if gene in deg_df['names'].values:
        x = deg_df.loc[deg_df['names'] == gene, 'logfoldchanges'].values[0]
        y = deg_df.loc[deg_df['names'] == gene, '-log10(padj)'].values[0]
        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시


plt.tight_layout()
plt.show()

In [ ]:
deg_df.to_csv("/home/jaehyunchoi/MASLD_collaboration/MP_analysis/DEG/scRNA_MASLD_DEG_markers.csv")

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='state',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='SLC7A5',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='SLC7A5',
    size=10
)

In [ ]:
adata_scmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

# Gene scoring

In [ ]:
gene_sets = {
    "cDC1_genes": ['XCR1', 'CLNK', 'TACSTD2', 'CLEC9A', 'EGLN3', 'IDO1', 'DNASE1L3', 'CADM1', 'S100B', 'KIAA0226L', 'WDFY4', 'CCND1', 'DUSP4', 'P2RY14', 'C1orf54', 'NRARP', 'CPNE3', 'SLAMF7', 'GPR157', 'B3GNT7', 'FLT3', 'NET1', 'RGCC', 'SERPINF1', 'BATF3', 'NDRG2', 'SNX3', 'ADAM28', 'AIM2', 'IL18R1', 'GPR171', 'PTPRCAP', 'IRF8', 'CPVL', 'C12orf45', 'TPD52', 'SLC38A1', 'LMNA', 'CD59', 'C15orf48', 'NFATC2', 'NAV1', 'MAP2K6', 'PTMS', 'ARHGAP5', 'HLA-DPB1', 'CST7', 'LGALS2', 'HLA-DQB1', 'PPA1', 'TCTN3', 'APOL1', 'CAMK2D', 'HLA-DPA1', 'CD74', 'HLA-DQA1', 'CST3', 'BCL2L11', 'DHRS3', 'ID2', 'RAB11FIP1', 'QPRT', 'OSBPL9', 'PTPN22', 'PTPN7', 'FAM129A', 'DST', 'FUCA1', 'ANPEP', 'MARCKSL1', 'HLA-DRB1', 'CLIC2', 'STK17A', 'RGS1', 'HLA-DRA', 'BASP1', 'ACTN1', 'PDLIM7', 'CD40', 'AC093673.5', 'NCOA7', 'PDE4DIP', 'TUBA1C', 'STMN1', 'CSF2RA', 'CD38', 'MIR4435-1HG', 'DUSP2', 'LGMN', 'PKIB', 'VMO1', 'SPATS2L'],
    "cDC2_genes": ['FCER1A', 'PPP1R14A', 'CD1C', 'AKR1C3', 'ENHO', 'CD1E', 'SLC38A1', 'P2RY14', 'AFF3', 'FCGR2B', 'CLEC10A', 'NDRG2', 'CST7', 'PKIB', 'CACNA2D3', 'ADAM28', 'SDPR', 'PLD4', 'NET1', 'FLT3', 'GAS6', 'ALOX5AP', 'AGPAT9', 'HLA-DQA1', 'RGS1', 'FILIP1L', 'C12orf45', 'HLA-DQB1', 'IL1R2', 'ARHGAP5', 'HLA-DPB1', 'GPR183', 'AXL', 'ADAM8', 'USP53', 'SERPINF1', 'RGCC', 'HLA-DPA1', 'CD72', 'GSN', 'CLIC2', 'ACAP1', 'ATP1B1', 'HLA-DRA', 'PMAIP1', 'CD74', 'CSF2RA', 'AREG', 'PPA1', 'FPR3', 'HLA-DRB1', 'ALCAM', 'SPATS2L', 'ARL4C', 'PTMS', 'STMN1', 'LMNA', 'VSIG4'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "MigcDC_genes": ['LAD1', 'CCR7', 'LAMP3', 'CCL19', 'CCL22', 'IL7R', 'SLC7A11', 'FSCN1', 'RAMP1', 'IDO1', 'CCL17', 'CD1B', 'MIR155HG', 'TNFRSF4', 'CXCL9', 'GPR157', 'DUSP4', 'IL4I1', 'EBI3', 'TRAF1', 'MARCKSL1', 'SPIB', 'ARHGAP10', 'HLA-DQB2', 'CD1E', 'MGLL', 'IL32', 'DUSP5', 'NRP2', 'CST7', 'PSD3', 'BIRC3', 'IRF4', 'C15orf48', 'HILPDA', 'GADD45A', 'SLC38A1', 'TRAF5', 'C12orf45', 'TXN', 'RAB9A', 'TBC1D4', 'TUBA1C', 'RGS1', 'SERPINF1', 'NMRK1', 'PPA1', 'CD40', 'GSN', 'POGLUT1', 'CLIC2', 'NAV1', 'PVRL2', 'PTGIR', 'RP11-138A9.2', 'FABP5', 'ALCAM', 'FAM129A', 'NET1', 'CSF2RA', 'ANTXR2', 'UGCG', 'GPR137B', 'CD1C', 'ADAM8', 'RELB', 'PALLD', 'SLAMF7', 'ST3GAL6', 'RP11-356I2.4', 'GPR183', 'HLA-DQA1', 'NFKB1', 'FLT3', 'RP11-138A9.1', 'CERS6', 'DNASE1L3', 'P2RY8', 'CD83', 'MAFF', 'SESN1', 'CYB5A', 'PTMS', 'SDC2', 'CCL5', 'ATP1B1', 'ETV3', 'ATF5', 'HLA-DQB1', 'FILIP1L', 'PMAIP1', 'PIM3', 'ID2', 'MARCKS', 'EPSTI1', 'BHLHE40', 'SOCS1', 'BASP1', 'HLA-DPB1', 'CD74', 'APOC1', 'HLA-DRA', 'CD72', 'GBP1', 'NR4A3', 'HLA-DPA1', 'PLEK', 'LMNA', 'CLEC10A'],
    "Monocyte_genes":['S100A8', 'S100A12', 'S100A9', 'VCAN', 'CTB-61M7.2', 'THBS1', 'EREG', 'MGST1', 'RP11-1143G9.4', 'ASGR1', 'BST1', 'AQP9', 'FCN1', 'CDA', 'LYZ', 'SLC2A3', 'MEGF9', 'APOBEC3A', 'CD300E', 'CD36', 'RAB27A', 'CLEC4E', 'TMEM176A', 'SELL', 'LGALS2', 'CXCL2', 'ITGAM', 'CSF3R', 'G0S2', 'EGR1', 'SOCS3', 'CCDC69', 'CAPG', 'TREM1', 'SLC11A1', 'AREG'],
    "Patmono_genes":['LYPD2', 'CDKN1C', 'CKB', 'PTP4A3', 'CD79B', 'HES4', 'PPM1N', 'SYTL1', 'FCGR3A', 'LILRA3', 'RHOC', 'RP11-290F20.3', 'TESC', 'SPN', 'TCF7L2', 'TPPP3', 'SLC2A6', 'SLC44A2', 'RRAS', 'SNX18', 'OAS1', 'FAM110A', 'VMO1', 'MTSS1', 'LILRB2', 'PHF19', 'TBC1D10C', 'FPR2', 'LILRA5', 'IFITM2', 'SIGLEC10', 'MYO1G', 'PAG1', 'SIDT2', 'LTB', 'LST1', 'S1PR4', 'ITGAL', 'STXBP2', 'CNIH4', 'GCH1', 'IFITM3', 'KLF2'],
    "LAM_genes":['FN1', 'CST6', 'SPP1', 'MT1G', 'TREM2', 'GPNMB', 'SLC16A10', 'CD9', 'EMP1', 'APOC1', 'LMNA', 'C15orf48', 'CLEC5A', 'IL7R', 'FABP5', 'RNASE1', 'FBP1', 'SDS', 'RAB13', 'OLR1', 'APOE', 'RGCC', 'GPX3', 'TSC22D1', 'SLC39A8', 'CAPG', 'ACP5', 'MMP19', 'ALCAM', 'GSN', 'FPR3', 'PDE4DIP', 'MT1X', 'LHFPL2', 'SDC2', 'MITF', 'PLA2G7', 'LGALS3', 'PHLDA1', 'ANKRD28', 'AVPI1', 'PTMS', 'MARCO', 'B3GNT5', 'PLTP', 'GPR183', 'DAB2', 'CD59', 'RGS1', 'LGMN'],
    "MATLAM_genes":['TREM2', 'GPNMB', 'CD9', 'RNASE1', 'OTOA', 'SDS', 'A2M', 'APOC1', 'LGMN', 'FCGBP', 'FABP5', 'SLCO2B1', 'APOE', 'C1QC', 'PLA2G7', 'FPR3', 'ID3', 'DAB2', 'AXL', 'C1QB', 'PLTP', 'RGS1', 'LTC4S', 'CD72', 'GATM', 'MGLL', 'LMNA', 'ABCC5', 'GAS6', 'ANKH', 'STMN1', 'C1QA', 'ALCAM', 'MSR1', 'PKIB', 'PHLDA1', 'ATP1B1', 'GPR34', 'PTMS', 'RGCC', 'MGAT4A', 'FCGR2B', 'LIPA', 'ACP5', 'CTSL', 'MITF', 'MAF', 'EBI3', 'KCNMA1', 'MERTK', 'VSIG4', 'C15orf48', 'FILIP1L', 'SEPP1', 'SLC40A1', 'CD81', 'GPR183', 'SDC2', 'GSN', 'FOLR2', 'CD59', 'CD84', 'CPM', 'MS4A4A', 'ABCA1', 'HLA-DQA1', 'CLEC10A', 'CLIC2', 'ARL4C', 'RGL1', 'EPB41L2', 'MMP19', 'AVPI1', 'ALB', 'C1orf54', 'PDK4', 'CD1C', 'VMO1'],
    "MAC1_genes":['IGSF21', 'FCGBP', 'USP53', 'ID3', 'C1QC', 'A2M', 'SDS', 'PLTP', 'MRC1L1', 'SLCO2B1', 'LTC4S', 'CD9', 'FCGR2B', 'RGS1', 'VSIG4', 'FPR3', 'C1QB', 'GPR34', 'AXL', 'FILIP1L', 'PKIB', 'C1QA', 'CD72', 'CLEC9A', 'NDRG2', 'CLEC10A', 'GATM', 'ALB', 'GPR183', 'MSR1', 'HLA-DPA1', 'PAPOLG', 'DAB2', 'PLD4', 'LPAR6', 'PHLDA1', 'EBI3', 'F13A1', 'FCER1A', 'HLA-DQA1', 'CPM', 'HLA-DPB1', 'ALOX5AP', 'SERPINF1', 'HLA-DRB1', 'RCAN1', 'STMN1', 'HLA-DRA', 'CCL3L1', 'ATP1B1', 'HLA-DQB1', 'PRDM1', 'MAF', 'ALCAM', 'C1orf54', 'APOC1', 'PTMS', 'LGALS3BP', 'CLIC2', 'RGCC', 'SLC40A1', 'VMO1', 'SERPING1', 'EPB41L2', 'CCL3', 'FOLR2', 'CD1C', 'PDK4', 'SEPP1', 'APOE'],
    "moKC_genes":['RNASE1', 'IGSF21', 'FCGBP', 'LILRB5', 'SLCO2B1', 'C1QC', 'SEPP1', 'PLTP', 'MRC1L1', 'NRP1', 'A2M', 'C1QB', 'TMEM37', 'LYVE1', 'FRMD4A', 'GFRA2', 'APOE', 'FOLR2', 'GPR34', 'OLFML2B', 'C1QA', 'LGMN', 'MAF', 'ID3', 'CXCL12', 'AXL', 'MERTK', 'SLC40A1', 'DAB2', 'MRC1', 'DHRS3', 'PDK4', 'GATM', 'LTC4S', 'MS4A4A', 'GPNMB', 'APOC1', 'USP53', 'EBI3', 'FPR3', 'FILIP1L', 'ALB', 'STAB1', 'SDC3', 'SIGLEC1', 'KCNMA1', 'VSIG4', 'ANKH', 'PKIB', 'MSR1', 'VCAM1', 'FCHO2', 'CCL3', 'SERPING1', 'CLEC9A', 'LGALS3BP', 'CCL4', 'CCL3L1', 'CD72', 'STMN1', 'CPM', 'CD163', 'RGL1', 'ABCA1', 'CD5L', 'EPS8', 'C2', 'CLIC2', 'RGS1', 'SPRED1', 'ATP1B1', 'TSPAN4', 'RCAN1', 'LIPA', 'SMPDL3A', 'WWP1', 'FRMD4B', 'EPB41L2', 'CD81', 'TCF4', 'AP2A2', 'SPATS2L', 'CD84', 'DST', 'FMNL2', 'MS4A7', 'CD59', 'CTSL', 'PLD3', 'HLA-DPA1', 'PTMS', 'ARL4C', 'VMO1', 'C1orf54', 'EGR2', 'CLEC10A', 'FCGR3A'],
    "KC_genes":['TIMD4', 'SLC16A9', 'NDST3', 'KCNAB1', 'ITGAD', 'CD5L', 'VCAM1', 'SELENBP1', 'BCAM', 'CDH5', 'CETP', 'RND3', 'FEZ1', 'FABP3', 'SDC3', 'CXCL12', 'ITLN1', 'CTD-2337J16.1', 'LYVE1', 'SEPP1', 'TMEM37', 'MARCO', 'SCD', 'LILRB5', 'PPAP2B', 'ITGA9', 'IGF1', 'APOE', 'SLC7A8', 'EPAS1', 'SLC40A1', 'FRMD4A', 'SUCNR1', 'ACVRL1', 'IGFBP4', 'ME1', 'LGMN', 'SIGLEC1', 'SIGLEC11', 'GPR126', 'FOLR2', 'LAG3', 'ITGB5', 'NRP1', 'C1QB', 'AXL', 'MRC1', 'C1QA', 'IFI27', 'C1QC', 'C2', 'SLC46A1', 'SLCO2B1', 'ETV5', 'SCN1B', 'NR1H3', 'DHRS3', 'PDK4', 'CD209', 'CCL4L1', 'GPNMB', 'ARHGEF12', 'CD59', 'RGL1', 'ANKRD36C', 'CCL4', 'MAF', 'EPB41L2', 'GATM', 'CCND1', 'CTSF', 'IFT74', 'LIPA', 'A2M', 'MYO9A', 'ATP1B1', 'CCL4L2', 'IFIT1', 'SERPING1', 'SPATS2L', 'MERTK', 'FUCA1', 'ADORA3', 'SMPDL3A', 'CCL3', 'ABCA1', 'MRC1L1', 'ANKH', 'CMKLR1', 'DST', 'PLD3', 'PLTP', 'FSCN1', 'APOC1', 'MMD', 'QPRT', 'VSIG4', 'RHOBTB3', 'EXT1', 'DAB2', 'ALB', 'RCAN1', 'FAM213A', 'EBI3', 'GFRA2', 'CD82', 'CCL3L3', 'PLA2G15', 'WWP1', 'CD81', 'ADM', 'ABCG1', 'FXYD6', 'STOM', 'KCNMA1', 'IFIT2', 'CD38', 'PPP1R12B', 'APOC3', 'CTSL', 'LGALS3BP', 'CR1', 'MSR1', 'TNFRSF21', 'CD163', 'GCNT1', 'EPHX1', 'SASH1', 'FILIP1L', 'SNX24', 'CLIC2', 'IFIT3', 'C1orf54', 'FPR3', 'FRMD4B', 'LTC4S', 'MMP19', 'CREG1', 'HMOX1', 'ZNF189', 'SDPR', 'NCEH1', 'FMNL2', 'CCL3L1', 'ITGAV', 'TSPAN4', 'ALDH1A1', 'NFIA', 'CTSB', 'PDE4DIP', 'ARL4C', 'ARRDC3', 'STMN1', 'RBP7', 'MS4A4A', 'EGR2', 'CD84', 'TCF4', 'VMO1', 'DSC2', 'GPR34', 'FCHO2', 'SLC9A9', 'PLA2G7', 'ADAP2', 'SCARB2', 'IL10', 'FCGRT', 'PHACTR2', 'AP2A2', 'ELL2', 'ACP2', 'MS4A7', 'P4HA1', 'PDGFC', 'GPR137B', 'SGPP1', 'SPRED1', 'PKIB', 'IFI44L', 'PRNP', 'CD72', 'SIAH2', 'MS4A6A', 'EPS8', 'UGCG', 'IDH1', 'BLVRB', 'ICAM1', 'RGS1', 'CTSD', 'CD63', 'FCGR3A', 'MT1F', 'GIMAP5', 'KCNJ2']
    }


In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["cDC1_genes"] , score_name="cDC1_score")
sc.pl.umap(adata_scmono, color='cDC1_score', cmap='bwr', title='cDC1_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["cDC2_genes"] , score_name="cDC2_score")
sc.pl.umap(adata_scmono, color='cDC2_score', cmap='bwr', title='cDC2_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["MigcDC_genes"] , score_name="MigcDC_score")
sc.pl.umap(adata_scmono, color='MigcDC_score', cmap='bwr', title='MigcDC_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["Monocyte_genes"] , score_name="Monocyte_score")
sc.pl.umap(adata_scmono, color='Monocyte_score', cmap='bwr', title='Monocyte_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["Patmono_genes"] , score_name="Patmono_score")
sc.pl.umap(adata_scmono, color='Patmono_score', cmap='bwr', title='Patmono_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["LAM_genes"] , score_name="LAM_score")
sc.pl.umap(adata_scmono, color='LAM_score', cmap='bwr', title='LAM_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["MATLAM_genes"] , score_name="MATLAM_score")
sc.pl.umap(adata_scmono, color='MATLAM_score', cmap='bwr', title='MATLAM_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["MAC1_genes"] , score_name="MAC1_score")
sc.pl.umap(adata_scmono, color='MAC1_score', cmap='bwr', title='MAC1_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["moKC_genes"] , score_name="moKC_score")
sc.pl.umap(adata_scmono, color='moKC_score', cmap='bwr', title='moKC_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["KC_genes"] , score_name="KC_score")
sc.pl.umap(adata_scmono, color='KC_score', cmap='bwr', title='KC_score')

In [ ]:
adata_scmono

In [ ]:
adata_scmono.raw

In [ ]:
sc.tl.leiden(adata_scmono, resolution=2, key_added="myl_cluster")
sc.pl.umap(adata_scmono, color=['myl_cluster'])

In [ ]:
sc.tl.leiden(adata_scmono, resolution=0.57, key_added="10_cluster")
sc.pl.umap(adata_scmono, color=['10_cluster'])

In [ ]:
celltypist.models.models_description()

In [ ]:
predictions_cluster = celltypist.annotate(adata_scmono, model='Healthy_Human_Liver.pkl', majority_voting=True, over_clustering=adata_scmono.obs["myl_cluster"]
)
adata_scmono =predictions_cluster.to_adata()
adata_scmono.obs['celltypist_liver'] = adata_scmono.obs['majority_voting']

In [ ]:
predictions_cluster = celltypist.annotate(adata_scmono, model='Immune_All_Low.pkl', majority_voting=True, over_clustering=adata_scmono.obs["myl_cluster"]
)
adata_scmono =predictions_cluster.to_adata()

In [ ]:
adata_scmono.obs

In [ ]:
adata_scmono.obs['Immune_low'] = adata_scmono.obs['majority_voting']

In [ ]:
predictions_cluster = celltypist.annotate(adata_scmono, model='Immune_All_High.pkl', majority_voting=True, over_clustering=adata_scmono.obs["myl_cluster"]
)
adata_scmono =predictions_cluster.to_adata()

In [ ]:
adata_scmono.obs['Immune_High'] = adata_scmono.obs['majority_voting']

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='Immune_low',
    #legend_loc='on data',
    legend_fontsize=12,
    frameon=False,
    title='Immune_celltypist',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=12,
    frameon=False,
    title='Immune_celltypist',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='Immune_High',
    #legend_loc='on data',
    legend_fontsize=12,
    frameon=False,
    title='Immune_celltypist',
    size=10
)

In [ ]:
adata_scmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='SLC7A5',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='MKI67_expression',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color=['CD4'],
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='MKI67_expression',
    size=10
)

# celltypist immune low analysis

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_scmono.obs['Immune_low']
batch_labels = adata_scmono.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
sc.pl.dotplot(adata_scmono,["SLC7A5","CCL2","NLRP3","TLR4"], groupby="Immune_low", standard_scale=None)

In [ ]:
sc.pl.dotplot(adata_scmono,["SLC7A5","CCL2","NLRP3","TLR4"], groupby="myl_cluster", standard_scale=None)

In [ ]:
adata_scmono

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'SLC7A5',groupby='myl_cluster',ax=ax)
plt.show()

In [ ]:
genes = ['SLC7A5', 'CCL2', 'NLRP3', 'TLR4', 'TNF', 'MTOR', 'AKT1', 'SREBF1']  # 예시 유전자 리스트
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(8, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='Immune_low',
        ax=axs[i],
        #palette = palette,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sc.tl.dendrogram(adata_scmono,groupby='Immune_low')
sc.tl.rank_genes_groups(adata_scmono, groupby='Immune_low', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scmono, n_genes=10)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata_scmono, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scmono,
    var_names=top_genes,
    groupby="Immune_low",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df

In [ ]:
adata_scmono.obs['Immune_low'].value_counts()

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata_scmono.obs['Immune_low'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata_scmono.obs[adata_scmono.obs['Immune_low'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata_scmono[selected_indices]

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata_scmono.obs['Immune_low'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata_scmono.obs[adata_scmono.obs['Immune_low'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata_scmono[selected_indices]
fig_dict = sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="Immune_low",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False,
    return_fig=True
)

print(type(fig_dict))
print(fig_dict.keys())

In [ ]:
fig_dict = sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="Immune_low",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False,
    return_fig=True
)

print(type(fig_dict))
print(fig_dict.keys())

In [ ]:
adata_scmono

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_scmono.obs['state']
cluster = adata_scmono.obs['Immune_low']

# 클러스터(세포타입) 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):  # 문자형 label일 경우 그냥 sorted() 사용
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "NAFLD") & (cluster == clus)).sum()
    ipf_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out) if (donor_in + donor_out) > 0 else 0
    ipf_frac = ipf_in / (ipf_in + ipf_out) if (ipf_in + ipf_out) > 0 else 0
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 기준 정렬 (선택사항)
df_result = df_result.sort_values("p_value")
print(df_result)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(15, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.39, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_scmono.obs['state']
cluster = adata_scmono.obs['Immune_High']

# 클러스터(세포타입) 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):  # 문자형 label일 경우 그냥 sorted() 사용
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "NAFLD") & (cluster == clus)).sum()
    ipf_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out) if (donor_in + donor_out) > 0 else 0
    ipf_frac = ipf_in / (ipf_in + ipf_out) if (ipf_in + ipf_out) > 0 else 0
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 기준 정렬 (선택사항)
df_result = df_result.sort_values("p_value")
print(df_result)

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.5, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_scmono.obs['state']
cluster = adata_scmono.obs['celltypist_liver']

# 클러스터(세포타입) 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):  # 문자형 label일 경우 그냥 sorted() 사용
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "NAFLD") & (cluster == clus)).sum()
    ipf_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out) if (donor_in + donor_out) > 0 else 0
    ipf_frac = ipf_in / (ipf_in + ipf_out) if (ipf_in + ipf_out) > 0 else 0
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 기준 정렬 (선택사항)
df_result = df_result.sort_values("p_value")
print(df_result)

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.66, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

# sub clustering 10 analysis

In [ ]:
adata_scmono

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='10_cluster',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='subclustering_10',
    size=10
)

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_scmono.obs['10_cluster']
batch_labels = adata_scmono.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'SLC7A5',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'PIK3CA',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'AKT1',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'MTOR',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'PTEN',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'RHEB',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'CTNNB1',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
sc.tl.dendrogram(adata_scmono,groupby='10_cluster')
sc.tl.rank_genes_groups(adata_scmono, groupby='10_cluster', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scmono, n_genes=10)

rank_df = sc.get.rank_genes_groups_df(adata_scmono, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scmono,
    var_names=top_genes,
    groupby="10_cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df.to_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scmono_DEG.csv")

In [ ]:
sc.tl.dendrogram(adata_scmono,groupby='10_cluster')
sc.tl.rank_genes_groups(adata_scmono, groupby='10_cluster', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scmono, n_genes=10)

rank_df = sc.get.rank_genes_groups_df(adata_scmono, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scmono,
    var_names=top_genes,
    groupby="10_cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:

from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_scmono.obs['state']
cluster = adata_scmono.obs['10_cluster']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters, key=int):  # 클러스터 번호 순 정렬
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "NAFLD") & (cluster == clus)).sum()
    ipf_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    ipf_frac = ipf_in / (ipf_in + ipf_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values("cluster")
print(df_result)
# cluster 열을 숫자형으로 바꾼 뒤 정렬
df_result['cluster'] = df_result['cluster'].astype(int)
df_result = df_result.sort_values('cluster')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.34, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
adata_scmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

In [ ]:
sc.pl.dotplot(adata_scmono, ['SLC7A5','PIK3CA','PIK3CB','PIK3CD','AKT1','MTOR','PTEN','RHEB','INPP4A','INPP4B'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['CD14','CCR2','S100A8','ITGAX','CLEC9A','CD1C','FLT3','CLEC4F','VSIG4','TIMD4'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['CLEC9A','FLT3','XCR1','CD1C','IRF4','CLEC10A'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['CLEC4F', 'VSIG4', 'MARCO', 'TIMD4', 'CD163',  'CCR2', 'S100A8', 'S100A9', 'FCN1',
                            'MARCO', 'TIMD4', 'CD163', 'CXCL9', 'CXCL10'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(7, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['CCR2','S100A8','S100A9','CD14','CLEC4F','VSIG4','MARCO','TIMD4'], 
              groupby="10_cluster",
              standard_scale=None,
              figsize=(7, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['WNT1','FZD1','LRP6','DVL2','AXIN1','APC','GSK3B','CTNNB1','LEF1'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(4, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['MAP1LC3B2','MAP1LC3A','COL1A1','FN1','SLC7A5'], 
              groupby="10_cluster",
              standard_scale=None,
              figsize=(4, 3))

---

# Part 2: scRNA_analysis-02

---

In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

# Total scRNA analysis

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_monoMAC_detail.h5ad")
adata_scrna

In [ ]:
adata_scrna.obs

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='majority_voting',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    #title='Cell type prediction',
    size=10
)

In [ ]:
adata_scrna.obs['majority_voting'].value_counts()

In [ ]:
adata_scrna.obs['cellannot'] = adata_scrna.obs['majority_voting'].replace({
    'T cells': 'Lymphoid',
    'Resident NK': 'Lymphoid',
    'Circulating NK/NKT': 'Lymphoid',
    'Macrophages': 'Myeloid',
    'Mono+mono derived cells': 'Myeloid',
    'Endothelial cells': 'Endothelial',
    'Fibroblasts': 'Stellete',
    'Hepatocytes': 'Hepatocyte',
    'Cholangiocytes' : 'Cholangiocyte'
})

sc.pl.umap(adata_scrna, color=['cellannot'])

In [ ]:
desired_order = ["Hepatocyte", "Cholangiocyte", "Endothelial", "Stellete", "Lymphoid", "Myeloid","B cells","Basophils","Plasma cells", "pDCs" ]
adata_scrna.obs['cellannot'] = pd.Categorical(adata_scrna.obs['cellannot'], categories=desired_order, ordered=True)

In [ ]:
sc.pl.umap(adata_scrna, color=['cellannot'])

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='cellannot',
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False, 
    return_fig=True
)
# 직접 저장
plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/cellseq_UMAP_v250112.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
adata_scrna.obs['state'] = adata_scrna.obs['state'].replace({
    'healthy': 'Healthy',
    'NAFLD': 'MASL'
})

sc.pl.umap(adata_scrna, color=['state'])

In [ ]:
desired_order = ["Healthy", "MASL"]
adata_scrna.obs['state'] = pd.Categorical(adata_scrna.obs['state'], categories=desired_order, ordered=True)

In [ ]:
adata_scrna.obs['sample'].value_counts()

In [ ]:
adata_scrna.obs['sample'] = adata_scrna.obs['sample'].replace({
    'Healthy1_Cd45p':'Healthy_1',
    'Healthy1_Cd45mA':'Healthy_1',
    'Healthy1_Cd45mB':'Healthy_1',
    'Healthy2_Cd45p':'Healthy_2',
    'Healthy2_Cd45m':'Healthy_2',
    'Healthy3_Cd45p':'Healthy_3',
    'Healthy3_Cd45mA':'Healthy_3',
    'Healthy3_Cd45mB':'Healthy_3',
    'Healthy4_Cd45m':'Healthy_4',
    'Healthy4_Cd45p':'Healthy_4',
    'Healthy5_Cd45p':'Healthy_5'
})

sc.pl.umap(adata_scrna, color=['sample'])

In [ ]:
sc.pl.dotplot(
    adata_scrna,
    ['SIRT1','PRKAA1','PRKAA2','EIF2AK4'],
    groupby="state",
    standard_scale='var'
)

In [ ]:
sc.pl.dotplot(
    adata_scrna,
    ['EIF2AK4','ATF4','SESN2','MIOS','SEC13','SEH1L','WDR24','WDR59','MTOR'],
    groupby="state",
    standard_scale=None
)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = adata_scrna.obs['cellannot']
batch_labels = adata_scrna.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(2, 5))

# 그래프 설정
plt.title('Major cell lineages in MASL vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASL vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = adata_scrna.obs['cellannot']
batch_labels = adata_scrna.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in MASL vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASL vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = adata_scrna.obs['cellannot']
batch_labels = adata_scrna.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in MASL vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASL vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
sc.tl.dendrogram(adata_scrna,groupby='cellannot')
sc.tl.rank_genes_groups(adata_scrna, groupby='cellannot', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scrna, n_genes=10)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata_scrna, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scrna,
    var_names=top_genes,
    groupby="cellannot",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df

In [ ]:
rank_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cell_total_DEG.csv")

In [ ]:
sc.pl.heatmap(
    adata_scrna,
    var_names=top_genes,
    groupby="cellannot",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=False, 
    #return_fig=True,
    figsize=(12, 7)
)
# 직접 저장
plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/cellseq_heatmap_v250112.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata_scrna.obs['cellannot'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata_scrna.obs[adata_scrna.obs['cellannot'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata_scrna[selected_indices]

sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="cellannot",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False
)

In [ ]:
sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="cellannot",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False
)
# 직접 저장
plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/cellseq_heatmap_bin_v250112.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
adata_scrna.obs['state']

In [ ]:
sc.tl.dendrogram(adata_scrna,groupby='state')
sc.tl.rank_genes_groups(adata_scrna, groupby='state', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scrna, n_genes=10)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata_scrna, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scrna,
    var_names=top_genes,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df

In [ ]:
rank_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cell_total_state_DEG.csv")

# scRNA monocyte analysis

In [ ]:
adata1=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono01.h5ad")
adata1

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")
adata

In [ ]:
adata.obs['monocluster'] = adata.obs['10_cluster'].replace({
    '0': 'mono0',
    '1': 'mono1',
    '2': 'mono2',
    '3': 'mono3',
    '4': 'mono4',
    '5': 'mono5',
    '6': 'mono6',
    '7': 'mono7',
    '8': 'mono8',
    '9': 'mono9'
})

sc.pl.umap(adata, color=['monocluster'])

In [ ]:
adata.obs.to_csv("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono_obs.csv")

In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.io import mmwrite

# AnnData 객체: adata
# adata.X         → expression matrix (희소 행렬 가능)
# adata.obs       → cell metadata
# adata.var       → gene metadata

# 1️⃣ Expression matrix 저장
mmwrite("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_matrix.mtx", csr_matrix(adata.X))

# 2️⃣ Cell metadata 저장
adata.obs.to_csv("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_cell_metadata.csv")

# 3️⃣ Gene metadata 저장
adata.var.to_csv("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_gene_metadata.csv")

In [ ]:
adata.obs['monoMAC'] = adata.obs['10_cluster'].replace({
    '0': 'mono1',
    '1': 'KC1',
    '2': 'cDC2',
    '3': 'mono2',
    '4': 'KC2',
    '5': 'mono3',
    '6': 'mono4',
    '7': 'KC3',
    '8': 'cDC1',
    '9': 'cDC3'
})

sc.pl.umap(adata, color=['monoMAC'])

In [ ]:
desired_order = ["mono1", "mono2","mono3","mono4","cDC1","cDC2","cDC3","KC1","KC2","KC3"]
adata.obs['monoMAC'] = pd.Categorical(adata.obs['monoMAC'], categories=desired_order, ordered=True)

In [ ]:
sc.pl.umap(
    adata,
    color='monoMAC',
    frameon=False,
    title='Cell type prediction',
    size=20
)

In [ ]:
gene_sets = {
    "cDC1_genes": ['XCR1', 'CLNK', 'TACSTD2', 'CLEC9A', 'EGLN3', 'IDO1', 'DNASE1L3', 'CADM1', 'S100B', 'KIAA0226L', 'WDFY4', 'CCND1', 'DUSP4', 'P2RY14', 'C1orf54', 'NRARP', 'CPNE3', 'SLAMF7', 'GPR157', 'B3GNT7', 'FLT3', 'NET1', 'RGCC', 'SERPINF1', 'BATF3', 'NDRG2', 'SNX3', 'ADAM28', 'AIM2', 'IL18R1', 'GPR171', 'PTPRCAP', 'IRF8', 'CPVL', 'C12orf45', 'TPD52', 'SLC38A1', 'LMNA', 'CD59', 'C15orf48', 'NFATC2', 'NAV1', 'MAP2K6', 'PTMS', 'ARHGAP5', 'HLA-DPB1', 'CST7', 'LGALS2', 'HLA-DQB1', 'PPA1', 'TCTN3', 'APOL1', 'CAMK2D', 'HLA-DPA1', 'CD74', 'HLA-DQA1', 'CST3', 'BCL2L11', 'DHRS3', 'ID2', 'RAB11FIP1', 'QPRT', 'OSBPL9', 'PTPN22', 'PTPN7', 'FAM129A', 'DST', 'FUCA1', 'ANPEP', 'MARCKSL1', 'HLA-DRB1', 'CLIC2', 'STK17A', 'RGS1', 'HLA-DRA', 'BASP1', 'ACTN1', 'PDLIM7', 'CD40', 'AC093673.5', 'NCOA7', 'PDE4DIP', 'TUBA1C', 'STMN1', 'CSF2RA', 'CD38', 'MIR4435-1HG', 'DUSP2', 'LGMN', 'PKIB', 'VMO1', 'SPATS2L'],
    "cDC2_genes": ['FCER1A', 'PPP1R14A', 'CD1C', 'AKR1C3', 'ENHO', 'CD1E', 'SLC38A1', 'P2RY14', 'AFF3', 'FCGR2B', 'CLEC10A', 'NDRG2', 'CST7', 'PKIB', 'CACNA2D3', 'ADAM28', 'SDPR', 'PLD4', 'NET1', 'FLT3', 'GAS6', 'ALOX5AP', 'AGPAT9', 'HLA-DQA1', 'RGS1', 'FILIP1L', 'C12orf45', 'HLA-DQB1', 'IL1R2', 'ARHGAP5', 'HLA-DPB1', 'GPR183', 'AXL', 'ADAM8', 'USP53', 'SERPINF1', 'RGCC', 'HLA-DPA1', 'CD72', 'GSN', 'CLIC2', 'ACAP1', 'ATP1B1', 'HLA-DRA', 'PMAIP1', 'CD74', 'CSF2RA', 'AREG', 'PPA1', 'FPR3', 'HLA-DRB1', 'ALCAM', 'SPATS2L', 'ARL4C', 'PTMS', 'STMN1', 'LMNA', 'VSIG4'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "MigcDC_genes": ['LAD1', 'CCR7', 'LAMP3', 'CCL19', 'CCL22', 'IL7R', 'SLC7A11', 'FSCN1', 'RAMP1', 'IDO1', 'CCL17', 'CD1B', 'MIR155HG', 'TNFRSF4', 'CXCL9', 'GPR157', 'DUSP4', 'IL4I1', 'EBI3', 'TRAF1', 'MARCKSL1', 'SPIB', 'ARHGAP10', 'HLA-DQB2', 'CD1E', 'MGLL', 'IL32', 'DUSP5', 'NRP2', 'CST7', 'PSD3', 'BIRC3', 'IRF4', 'C15orf48', 'HILPDA', 'GADD45A', 'SLC38A1', 'TRAF5', 'C12orf45', 'TXN', 'RAB9A', 'TBC1D4', 'TUBA1C', 'RGS1', 'SERPINF1', 'NMRK1', 'PPA1', 'CD40', 'GSN', 'POGLUT1', 'CLIC2', 'NAV1', 'PVRL2', 'PTGIR', 'RP11-138A9.2', 'FABP5', 'ALCAM', 'FAM129A', 'NET1', 'CSF2RA', 'ANTXR2', 'UGCG', 'GPR137B', 'CD1C', 'ADAM8', 'RELB', 'PALLD', 'SLAMF7', 'ST3GAL6', 'RP11-356I2.4', 'GPR183', 'HLA-DQA1', 'NFKB1', 'FLT3', 'RP11-138A9.1', 'CERS6', 'DNASE1L3', 'P2RY8', 'CD83', 'MAFF', 'SESN1', 'CYB5A', 'PTMS', 'SDC2', 'CCL5', 'ATP1B1', 'ETV3', 'ATF5', 'HLA-DQB1', 'FILIP1L', 'PMAIP1', 'PIM3', 'ID2', 'MARCKS', 'EPSTI1', 'BHLHE40', 'SOCS1', 'BASP1', 'HLA-DPB1', 'CD74', 'APOC1', 'HLA-DRA', 'CD72', 'GBP1', 'NR4A3', 'HLA-DPA1', 'PLEK', 'LMNA', 'CLEC10A'],
    "Monocyte_genes":['S100A8', 'S100A12', 'S100A9', 'VCAN', 'CTB-61M7.2', 'THBS1', 'EREG', 'MGST1', 'RP11-1143G9.4', 'ASGR1', 'BST1', 'AQP9', 'FCN1', 'CDA', 'LYZ', 'SLC2A3', 'MEGF9', 'APOBEC3A', 'CD300E', 'CD36', 'RAB27A', 'CLEC4E', 'TMEM176A', 'SELL', 'LGALS2', 'CXCL2', 'ITGAM', 'CSF3R', 'G0S2', 'EGR1', 'SOCS3', 'CCDC69', 'CAPG', 'TREM1', 'SLC11A1', 'AREG'],
    "Patmono_genes":['LYPD2', 'CDKN1C', 'CKB', 'PTP4A3', 'CD79B', 'HES4', 'PPM1N', 'SYTL1', 'FCGR3A', 'LILRA3', 'RHOC', 'RP11-290F20.3', 'TESC', 'SPN', 'TCF7L2', 'TPPP3', 'SLC2A6', 'SLC44A2', 'RRAS', 'SNX18', 'OAS1', 'FAM110A', 'VMO1', 'MTSS1', 'LILRB2', 'PHF19', 'TBC1D10C', 'FPR2', 'LILRA5', 'IFITM2', 'SIGLEC10', 'MYO1G', 'PAG1', 'SIDT2', 'LTB', 'LST1', 'S1PR4', 'ITGAL', 'STXBP2', 'CNIH4', 'GCH1', 'IFITM3', 'KLF2'],
    "LAM_genes":['FN1', 'CST6', 'SPP1', 'MT1G', 'TREM2', 'GPNMB', 'SLC16A10', 'CD9', 'EMP1', 'APOC1', 'LMNA', 'C15orf48', 'CLEC5A', 'IL7R', 'FABP5', 'RNASE1', 'FBP1', 'SDS', 'RAB13', 'OLR1', 'APOE', 'RGCC', 'GPX3', 'TSC22D1', 'SLC39A8', 'CAPG', 'ACP5', 'MMP19', 'ALCAM', 'GSN', 'FPR3', 'PDE4DIP', 'MT1X', 'LHFPL2', 'SDC2', 'MITF', 'PLA2G7', 'LGALS3', 'PHLDA1', 'ANKRD28', 'AVPI1', 'PTMS', 'MARCO', 'B3GNT5', 'PLTP', 'GPR183', 'DAB2', 'CD59', 'RGS1', 'LGMN'],
    "MATLAM_genes":['TREM2', 'GPNMB', 'CD9', 'RNASE1', 'OTOA', 'SDS', 'A2M', 'APOC1', 'LGMN', 'FCGBP', 'FABP5', 'SLCO2B1', 'APOE', 'C1QC', 'PLA2G7', 'FPR3', 'ID3', 'DAB2', 'AXL', 'C1QB', 'PLTP', 'RGS1', 'LTC4S', 'CD72', 'GATM', 'MGLL', 'LMNA', 'ABCC5', 'GAS6', 'ANKH', 'STMN1', 'C1QA', 'ALCAM', 'MSR1', 'PKIB', 'PHLDA1', 'ATP1B1', 'GPR34', 'PTMS', 'RGCC', 'MGAT4A', 'FCGR2B', 'LIPA', 'ACP5', 'CTSL', 'MITF', 'MAF', 'EBI3', 'KCNMA1', 'MERTK', 'VSIG4', 'C15orf48', 'FILIP1L', 'SEPP1', 'SLC40A1', 'CD81', 'GPR183', 'SDC2', 'GSN', 'FOLR2', 'CD59', 'CD84', 'CPM', 'MS4A4A', 'ABCA1', 'HLA-DQA1', 'CLEC10A', 'CLIC2', 'ARL4C', 'RGL1', 'EPB41L2', 'MMP19', 'AVPI1', 'ALB', 'C1orf54', 'PDK4', 'CD1C', 'VMO1'],
    "MAC1_genes":['IGSF21', 'FCGBP', 'USP53', 'ID3', 'C1QC', 'A2M', 'SDS', 'PLTP', 'MRC1L1', 'SLCO2B1', 'LTC4S', 'CD9', 'FCGR2B', 'RGS1', 'VSIG4', 'FPR3', 'C1QB', 'GPR34', 'AXL', 'FILIP1L', 'PKIB', 'C1QA', 'CD72', 'CLEC9A', 'NDRG2', 'CLEC10A', 'GATM', 'ALB', 'GPR183', 'MSR1', 'HLA-DPA1', 'PAPOLG', 'DAB2', 'PLD4', 'LPAR6', 'PHLDA1', 'EBI3', 'F13A1', 'FCER1A', 'HLA-DQA1', 'CPM', 'HLA-DPB1', 'ALOX5AP', 'SERPINF1', 'HLA-DRB1', 'RCAN1', 'STMN1', 'HLA-DRA', 'CCL3L1', 'ATP1B1', 'HLA-DQB1', 'PRDM1', 'MAF', 'ALCAM', 'C1orf54', 'APOC1', 'PTMS', 'LGALS3BP', 'CLIC2', 'RGCC', 'SLC40A1', 'VMO1', 'SERPING1', 'EPB41L2', 'CCL3', 'FOLR2', 'CD1C', 'PDK4', 'SEPP1', 'APOE'],
    "moKC_genes":['RNASE1', 'IGSF21', 'FCGBP', 'LILRB5', 'SLCO2B1', 'C1QC', 'SEPP1', 'PLTP', 'MRC1L1', 'NRP1', 'A2M', 'C1QB', 'TMEM37', 'LYVE1', 'FRMD4A', 'GFRA2', 'APOE', 'FOLR2', 'GPR34', 'OLFML2B', 'C1QA', 'LGMN', 'MAF', 'ID3', 'CXCL12', 'AXL', 'MERTK', 'SLC40A1', 'DAB2', 'MRC1', 'DHRS3', 'PDK4', 'GATM', 'LTC4S', 'MS4A4A', 'GPNMB', 'APOC1', 'USP53', 'EBI3', 'FPR3', 'FILIP1L', 'ALB', 'STAB1', 'SDC3', 'SIGLEC1', 'KCNMA1', 'VSIG4', 'ANKH', 'PKIB', 'MSR1', 'VCAM1', 'FCHO2', 'CCL3', 'SERPING1', 'CLEC9A', 'LGALS3BP', 'CCL4', 'CCL3L1', 'CD72', 'STMN1', 'CPM', 'CD163', 'RGL1', 'ABCA1', 'CD5L', 'EPS8', 'C2', 'CLIC2', 'RGS1', 'SPRED1', 'ATP1B1', 'TSPAN4', 'RCAN1', 'LIPA', 'SMPDL3A', 'WWP1', 'FRMD4B', 'EPB41L2', 'CD81', 'TCF4', 'AP2A2', 'SPATS2L', 'CD84', 'DST', 'FMNL2', 'MS4A7', 'CD59', 'CTSL', 'PLD3', 'HLA-DPA1', 'PTMS', 'ARL4C', 'VMO1', 'C1orf54', 'EGR2', 'CLEC10A', 'FCGR3A'],
    "KC_genes":['TIMD4', 'SLC16A9', 'NDST3', 'KCNAB1', 'ITGAD', 'CD5L', 'VCAM1', 'SELENBP1', 'BCAM', 'CDH5', 'CETP', 'RND3', 'FEZ1', 'FABP3', 'SDC3', 'CXCL12', 'ITLN1', 'CTD-2337J16.1', 'LYVE1', 'SEPP1', 'TMEM37', 'MARCO', 'SCD', 'LILRB5', 'PPAP2B', 'ITGA9', 'IGF1', 'APOE', 'SLC7A8', 'EPAS1', 'SLC40A1', 'FRMD4A', 'SUCNR1', 'ACVRL1', 'IGFBP4', 'ME1', 'LGMN', 'SIGLEC1', 'SIGLEC11', 'GPR126', 'FOLR2', 'LAG3', 'ITGB5', 'NRP1', 'C1QB', 'AXL', 'MRC1', 'C1QA', 'IFI27', 'C1QC', 'C2', 'SLC46A1', 'SLCO2B1', 'ETV5', 'SCN1B', 'NR1H3', 'DHRS3', 'PDK4', 'CD209', 'CCL4L1', 'GPNMB', 'ARHGEF12', 'CD59', 'RGL1', 'ANKRD36C', 'CCL4', 'MAF', 'EPB41L2', 'GATM', 'CCND1', 'CTSF', 'IFT74', 'LIPA', 'A2M', 'MYO9A', 'ATP1B1', 'CCL4L2', 'IFIT1', 'SERPING1', 'SPATS2L', 'MERTK', 'FUCA1', 'ADORA3', 'SMPDL3A', 'CCL3', 'ABCA1', 'MRC1L1', 'ANKH', 'CMKLR1', 'DST', 'PLD3', 'PLTP', 'FSCN1', 'APOC1', 'MMD', 'QPRT', 'VSIG4', 'RHOBTB3', 'EXT1', 'DAB2', 'ALB', 'RCAN1', 'FAM213A', 'EBI3', 'GFRA2', 'CD82', 'CCL3L3', 'PLA2G15', 'WWP1', 'CD81', 'ADM', 'ABCG1', 'FXYD6', 'STOM', 'KCNMA1', 'IFIT2', 'CD38', 'PPP1R12B', 'APOC3', 'CTSL', 'LGALS3BP', 'CR1', 'MSR1', 'TNFRSF21', 'CD163', 'GCNT1', 'EPHX1', 'SASH1', 'FILIP1L', 'SNX24', 'CLIC2', 'IFIT3', 'C1orf54', 'FPR3', 'FRMD4B', 'LTC4S', 'MMP19', 'CREG1', 'HMOX1', 'ZNF189', 'SDPR', 'NCEH1', 'FMNL2', 'CCL3L1', 'ITGAV', 'TSPAN4', 'ALDH1A1', 'NFIA', 'CTSB', 'PDE4DIP', 'ARL4C', 'ARRDC3', 'STMN1', 'RBP7', 'MS4A4A', 'EGR2', 'CD84', 'TCF4', 'VMO1', 'DSC2', 'GPR34', 'FCHO2', 'SLC9A9', 'PLA2G7', 'ADAP2', 'SCARB2', 'IL10', 'FCGRT', 'PHACTR2', 'AP2A2', 'ELL2', 'ACP2', 'MS4A7', 'P4HA1', 'PDGFC', 'GPR137B', 'SGPP1', 'SPRED1', 'PKIB', 'IFI44L', 'PRNP', 'CD72', 'SIAH2', 'MS4A6A', 'EPS8', 'UGCG', 'IDH1', 'BLVRB', 'ICAM1', 'RGS1', 'CTSD', 'CD63', 'FCGR3A', 'MT1F', 'GIMAP5', 'KCNJ2']
    }
#sc.tl.score_genes(Liver_Myl_MASL, gene_list=gene_sets["Monocyte_genes"] , score_name="Monocyte_score")
#sc.pl.umap(Liver_Myl_MASL, color='Monocyte_score', cmap='bwr', title='Monocyte_score')

In [ ]:
sc.tl.score_genes(adata, gene_list=gene_sets["LAM_genes"] , score_name="lam_score")
sc.pl.umap(adata, color='lam_score', cmap='bwr', title='lam_score')

In [ ]:
sc.tl.score_genes(adata, gene_list=gene_sets["MAC1_genes"], score_name="mac_score")
sc.pl.umap(adata, color='mac_score', cmap='bwr', title='mac_score')

In [ ]:
SAM_gene = {'REM2', 'SPP1', 'CD63', 'FABP5', 'CD9'}
sc.tl.score_genes(adata, gene_list=SAM_gene , score_name="sam_score")
sc.pl.umap(adata, color='sam_score', cmap='bwr', title='sam_score')

In [ ]:
sc.pl.umap(
    adata,
    color=['REM2', 'SPP1', 'CD63'],
    frameon=False,
    #title='SLC7A5 expression',
    size=20
)

In [ ]:
sc.pl.umap(
    adata,
    color=['FABP5', 'CD9'],
    frameon=False,
    #title='SLC7A5 expression',
    size=20
)

In [ ]:
sc.pl.violin(adata, 'TREM2', groupby = 'monoMAC')

In [ ]:
sc.pl.dotplot(adata, 'TREM2', groupby = 'monoMAC')

In [ ]:
sc.pl.umap(
    adata,
    color='SLC7A5',
    frameon=False,
    title='SLC7A5 expression',
    size=20
)

In [ ]:
adata.obs['Myeloid_cluster'] = adata.obs['monoMAC'].replace({
    'mono1':'monocyte-derived',
    'mono2':'monocyte-derived',
    'mono3':'monocyte-derived',
    'mono4':'monocyte-derived',
    'KC1':'Kupffer_cells',
    'KC2':'Kupffer_cells',
    'KC3':'Kupffer_cells',
    'cDC1': 'conventional_DC',
    'cDC2': 'conventional_DC'
})

sc.pl.umap(adata, color=['Myeloid_cluster'])

In [ ]:
adata.obs['state'] = adata.obs['state'].replace({
    'healthy': 'Healthy',
    'NAFLD': 'MASL'
})

sc.pl.umap(adata, color=['state'])

In [ ]:
myl_genes = {
    'mono':['ITGAM','CCR2','CX3CR1','S100A8'],
    'cDC': ['ITGAX','XCR1','ZBTB46','BATF3','IRF4'],
    'KC': ['TIMD4','MARCO','VSIG4']}

In [ ]:
sc.pl.dotplot(adata, myl_genes, 
              groupby="Myeloid_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.matrixplot(adata, myl_genes, 
              groupby="Myeloid_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.matrixplot(adata, myl_genes, 
              groupby="monoMAC",
              standard_scale=None,
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata, ['LY6G6C','ITGAM','CCR2','CX3CR1','S100A8','CLEC4F','TIMD4','MARCO','VSIG4','ITGAX','XCR1','ZBTB46','BATF3','IRF4'], 
              groupby="Myeloid_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
genes = ['SLC7A5','LAT2','SLC43A1','SLC43A2','SLC3A2']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='monocluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CSF3R', 'IL1RN', 'CXCL8', 'C5AR1', 'TREM1','FPR1']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='monocluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['CD163', 'MRC1', 'MSR1', 'MARCO','TREM2','CD9','SPP1']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='monocluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))

sc.pl.violin(
    adata,
    keys='SLC7A5',
    groupby='monocluster',
    ax=ax,
    show=False,
    stripplot=True,
    jitter=True
)

plt.tight_layout()
plt.show()

In [ ]:
adata.obs['cellannot'] = adata.obs['monoMAC'].replace({
    'mono1': 'Classical Monocytes',
    'mono2': 'Patrolling Monocytes',
    'mono3': 'SLC7A5 high Monocytes',
    'mono4': 'CD3D noise',
    'KC1': 'KCs',
    'KC2': 'TREM2+ LAM',
    'KC3': 'moKCs',
    'cDC1': 'cDC1s',
    'cDC2': 'cDC2s',
    'cDC3': 'Migratory cDCs'
})

sc.pl.umap(adata, color=['cellannot'])

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

def fisher_test_score(adata, gene, groupby="monocluster"):
    
    # 1️⃣ expression 벡터 추출
    expr = adata[:, gene].X
    
    if hasattr(expr, "A"):  # sparse matrix
        expr = expr.A.flatten()
    else:
        expr = expr.flatten()
    
    # 2️⃣ group 정보
    groups = adata.obs[groupby].values
    
    # 3️⃣ median 기준 high / low
    median = np.median(expr)
    high = expr > median
    
    # 4️⃣ mono1 vs mono3만 사용
    mask = np.isin(groups, ["mono1", "mono5"])
    
    high = high[mask]
    groups = groups[mask]
    
    # 5️⃣ contingency table
    table = pd.crosstab(high, groups)
    
    # ensure 2x2
    if table.shape != (2,2):
        return 1.0
    
    _, p = fisher_exact(table)
    return p

In [ ]:
genes = ['SLC7A5','CCL2','NLRP3','TLR4','TNF','AKT1','SREBF1']
adata_sub = adata[adata.obs['monocluster'].isin(['mono1', 'mono5'])].copy()

n_gene = len(genes)
fig, axs = plt.subplots(n_gene, 1, figsize=(3, 2 * n_gene))

if n_gene == 1:
    axs = [axs]

for i, gene in enumerate(genes):

    sc.pl.violin(
        adata_sub,
        keys=gene,
        groupby='monocluster',
        ax=axs[i],
        show=False,
        stripplot=True,
        jitter=True
    )

    # Fisher test
    p = fisher_test_score(adata_sub, gene)
    star = p_to_star(p)

    # ⭐ expression 값 직접 추출
    expr = adata_sub[:, gene].X
    if hasattr(expr, "A"):
        expr = expr.A.flatten()
    else:
        expr = expr.flatten()

    ymax = np.max(expr)

    axs[i].text(
        0.5,
        0.95,   # 축 기준 위치 (더 안정적)
        star,
        ha="center",
        va="top",
        fontsize=12,
        fontweight="bold",
        transform=axs[i].transAxes
    )

    axs[i].set_title(gene)

In [ ]:
sc.pl.dotplot(adata, ['CCL2','NLRP3','TLR4','TNF','AKT1','SREBF1'], 
              groupby="monoMAC",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata, ['LY6G6C','ITGAM','CCR2','CX3CR1','S100A8','CLEC4F','TIMD4','MARCO','VSIG4','ITGAX','XCR1','ZBTB46','BATF3','IRF4'], 
              groupby="monoMAC",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata, ['FCGR3A','CD14'], 
              groupby="monoMAC",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata, ['CCR2','SPP1','CD14'], 
              groupby="cellannot",
              standard_scale=None,
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata, ['VSIG4', 'CLEC4F'], 
              groupby="monoMAC",
              #standard_scale='var',
              figsize=(3, 3))

In [ ]:
sc.pl.dotplot(adata, ['CST3', 'AIF1', 'HLA-DRA', 'LYZ', 'SAT1', 'CD68', 'CD14', 'FTL', 'TYROBP', 'CTSS', 'FCER1G','CD3D', 'CD3E', 'CD2', 'FCER1G'], 
              groupby="monoMAC",
              #standard_scale='var',
              figsize=(10, 3))

In [ ]:
sc.pl.umap(
    adata,
    color='CD8A',
    frameon=False,
    title='SLC7A5 expression',
    size=20
)

In [ ]:
sc.pl.dotplot(adata, ['SLC7A5','SLC43A2'], 
              groupby="monoMAC",
              standard_scale='var',
              figsize=(2.5, 3))

In [ ]:
sc.pl.dotplot(adata, ['SLC7A5','SLC43A2'], 
              groupby="state",
              standard_scale='var',
              figsize=(2.5, 3))

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata.obs['state']
cluster = adata.obs['monocluster']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):
    # 각 조건별 개수 계산
    donor_in = ((state == "Healthy") & (cluster == clus)).sum()
    donor_out = ((state == "Healthy") & (cluster != clus)).sum()
    masl_in = ((state == "MASL") & (cluster == clus)).sum()
    masl_out = ((state == "MASL") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [masl_in, masl_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    masl_frac = masl_in / (masl_in + masl_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "masl_fraction": masl_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values('cluster')
print(df_result)
df_result = df_result.sort_values('cluster')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='Healthy')
plt.bar(x + width/2, df_result['masl_fraction'], width, label='MASL')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('Myeloid cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['masl_fraction'][i])
        plt.text(x[i], 0.34, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata.obs['state']
cluster = adata.obs['monoMAC']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):
    # 각 조건별 개수 계산
    donor_in = ((state == "Healthy") & (cluster == clus)).sum()
    donor_out = ((state == "Healthy") & (cluster != clus)).sum()
    masl_in = ((state == "MASL") & (cluster == clus)).sum()
    masl_out = ((state == "MASL") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [masl_in, masl_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    masl_frac = masl_in / (masl_in + masl_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "masl_fraction": masl_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values('cluster')
print(df_result)
df_result = df_result.sort_values('cluster')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='Healthy')
plt.bar(x + width/2, df_result['masl_fraction'], width, label='MASL')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('Myeloid cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['masl_fraction'][i])
        plt.text(x[i], 0.34, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
adata.obs['state']

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata.obs['state']
cluster = adata.obs['cellannot']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):
    # 각 조건별 개수 계산
    donor_in = ((state == "Healthy") & (cluster == clus)).sum()
    donor_out = ((state == "Healthy") & (cluster != clus)).sum()
    masl_in = ((state == "MASL") & (cluster == clus)).sum()
    masl_out = ((state == "MASL") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [masl_in, masl_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    masl_frac = masl_in / (masl_in + masl_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "masl_fraction": masl_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values('cluster')
print(df_result)
df_result = df_result.sort_values('cluster')
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(15, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='Healthy')
plt.bar(x + width/2, df_result['masl_fraction'], width, label='MASL')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('Myeloid cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['masl_fraction'][i])
        plt.text(x[i], 0.34, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = adata.obs['monoMAC']
batch_labels = adata.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in sample bathc')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
desired_order = ["Healthy", "MASL"]
adata.obs['state'] = pd.Categorical(adata.obs['state'], categories=desired_order, ordered=True)
cluster_labels = adata.obs['monoMAC']
batch_labels = adata.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(2, 5))

# 그래프 설정
plt.title('Major cell lineages in sample bathc')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
desired_order = ["Healthy", "MASL"]
adata.obs['state'] = pd.Categorical(adata.obs['state'], categories=desired_order, ordered=True)
cluster_labels = adata.obs['monocluster']
batch_labels = adata.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(2, 5))

# 그래프 설정
plt.title('Major cell lineages in sample bathc')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
adata.obs

In [ ]:
sc.tl.dendrogram(adata,groupby='monoMAC')
sc.tl.rank_genes_groups(adata, groupby='monoMAC', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10)

rank_df = sc.get.rank_genes_groups_df(adata, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata,
    var_names=top_genes,
    groupby="monoMAC",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
sc.tl.dendrogram(adata,groupby='monocluster')
sc.tl.rank_genes_groups(adata, groupby='monocluster', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10)

rank_df = sc.get.rank_genes_groups_df(adata, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata,
    var_names=top_genes,
    groupby="monocluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata.obs['monocluster'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata.obs[adata.obs['monocluster'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata[selected_indices]

In [ ]:
sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=20,
    groupby="monocluster",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False
)

In [ ]:
adata.obs['monoMAC'].value_counts()

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata.obs['monoMAC'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata.obs[adata.obs['monoMAC'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata[selected_indices]

In [ ]:
desired_order = ["mono1", "mono2","mono3","mono4","cDC1","cDC2","cDC3","KC1","KC2","KC3"]
adata_sampled.obs['monoMAC'] = pd.Categorical(adata_sampled.obs['monoMAC'], categories=desired_order, ordered=True)

In [ ]:
sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="monoMAC",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False
)

In [ ]:
rank_df

In [ ]:
rank_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cell_monoMAC_DEG.csv")

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='monoMAC',groups=['mono3'],reference = 'rest', method='wilcoxon')
rank_mono3_df = sc.get.rank_genes_groups_df(adata, group='mono3')
rank_mono3_df

In [ ]:
rank_mono3_df.head(10)

In [ ]:
adata

In [ ]:
adata

In [ ]:
genes = ['SLC1A1','SLC1A2','SLC1A4','SLC1A5','SLC7A2','SLC7A5','SLC7A6','SLC7A7','SLC7A8','SLC7A9','SLC7A11',
         'SLC16A10','SLC25A12', 'SLC25A13','SLC25A18','SLC25A22','SLC25A44','SLC36A1','SLC36A4','SLC38A1',
         'SLC38A2','SLC38A3','SLC38A4','SLC38A6','SLC38A7','SLC38A9','SLC38A10','SLC43A1','SLC43A2',]

In [ ]:
sc.pl.matrixplot(adata, genes, 
              groupby="monocluster",
              standard_scale='var',
              figsize=(10, 3))

In [ ]:
import gseapy as gp
import scanpy as sc

In [ ]:
adata.obs.groupby('monocluster')['state'].value_counts()

In [ ]:
import gseapy as gp

libs = gp.get_library_name()
print(libs)

In [ ]:
adata.obs['state'] = pd.Categorical(adata.obs['state'], categories=["MASL", "Healthy"], ordered=True)
indices = adata.obs.sort_values(['monocluster', 'state']).index
adata = adata[indices,:]

In [ ]:
adata.obs["mono5_vs_other"] = adata.obs.monocluster.apply(
    lambda x: "mono5" if x == "mono5" else "other"
)

In [ ]:
adata.obs["mono5_vs_other"] = pd.Categorical(
    adata.obs["mono5_vs_other"],
    categories=["mono5", "other"],  # 순서 중요
    ordered=True
)

In [ ]:
cls = adata.obs["mono5_vs_other"].astype(str).values.tolist()

In [ ]:
set(cls)

In [ ]:
res = gp.gsea(
    data=adata.to_df().T,
    gene_sets="KEGG_2021_Human",
    cls=cls,
    permutation_num=1000,
    permutation_type='phenotype',
    outdir=None,
    method='s2n',
    threads=16
)

In [ ]:
res.res2d.head(10)

In [ ]:
res.ranking.shape

In [ ]:
res.res2d.head(20)

In [ ]:
term = res.res2d.Term
# gp.gseaplot(res.ranking, term=term[i], **res.results[term[i]])
axs = res.plot(terms=term[4])

In [ ]:
## Heatmap of gene expression
i = 4
genes = res.res2d.Lead_genes.iloc[i].split(";")
ax = gp.heatmap(df = res.heatmat.loc[genes],
           z_score=None,
           title=res.res2d.Term.iloc[i],
           figsize=(6,5),
           cmap=plt.cm.viridis,
           xticklabels=False)

In [ ]:
res.res2d[res.res2d["Term"].str.contains("mTOR", case=False)]

In [ ]:
term = "mTOR signaling pathway"

axs = res.plot(terms=term)

In [ ]:
res.res2d[res.res2d["Term"].str.contains("PI3K", case=False)]

In [ ]:
term = "PI3K-Akt signaling pathway"

axs = res.plot(terms=term)

In [ ]:
adata.obs.groupby(["sample", "monocluster"])["mTOR_score"].mean()

In [ ]:
import pandas as pd
import numpy as np

mono5 = adata[adata.obs.monocluster == "mono5"].copy()

expr = mono5.to_df()
expr["sample"] = mono5.obs["sample"].values
expr["state"] = mono5.obs["state"].values

# sample 단위 평균
pseudobulk = expr.groupby(["sample", "state"]).mean()

# state 분리
disease = pseudobulk.xs("MASL", level="state")
control = pseudobulk.xs("Healthy", level="state")

In [ ]:
from scipy.stats import ttest_ind

results = []

for gene in disease.columns:
    stat, pval = ttest_ind(disease[gene], control[gene], equal_var=False)
    logfc = disease[gene].mean() - control[gene].mean()
    results.append([gene, logfc, pval])

de = pd.DataFrame(results, columns=["gene", "logFC", "pval"])

In [ ]:
sig_genes = de[(de.pval < 0.05) & (de.logFC > 0)]["gene"]

In [ ]:
import gseapy as gp

kegg = gp.get_library(name="KEGG_2021_Human")

mtor_genes = kegg["mTOR signaling pathway"]

In [ ]:
from scipy.stats import fisher_exact

# universe
all_genes = set(de["gene"])

# 교집합
overlap = len(set(sig_genes) & set(mtor_genes))

# contingency table
a = overlap
b = len(sig_genes) - a
c = len(mtor_genes) - a
d = len(all_genes) - (a + b + c)

table = [[a, b],
         [c, d]]

oddsratio, pvalue = fisher_exact(table)

print("Fisher p-value:", pvalue)
print("Odds ratio:", oddsratio)

In [ ]:
adata.write("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono03.h5ad")

# Monocle3 trajectory analysis

In [ ]:
adata_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono01.h5ad")
adata_raw

In [ ]:
adata_raw.obs

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono03.h5ad")
adata

In [ ]:
adata_raw.obs['monoMAC']=adata.obs['monoMAC']

In [ ]:
adata.obs['monoMAC'].value_counts()

In [ ]:
adata_filter = adata[adata.obs['monoMAC'].isin(['mono1','mono2','mono3','KC1','KC2','KC3'])].copy()
adata_filter.obs['monoMAC'].value_counts()

In [ ]:
import pandas as pd
from scipy.io import mmwrite
from scipy.sparse import csr_matrix
mmwrite("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_matrix_raw.mtx", adata_raw.X)
adata_raw.obs.to_csv("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_cell_metadata_raw.csv")
adata_raw.var.to_csv("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_gene_metadata_raw.csv")
mmwrite("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_csr_matrix_raw.mtx", csr_matrix(adata_raw.X))

In [ ]:
import pandas as pd

umap_df = pd.DataFrame(
    adata.obsm["X_umap"],
    index=adata.obs_names,          # ⭐ cell 이름 필수
    columns=["UMAP1", "UMAP2"]
)

umap_df.to_csv("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scanpy_umap.csv")

In [ ]:
import pandas as pd
from scipy.io import mmwrite
from scipy.sparse import csr_matrix
mmwrite("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_matrix_filter.mtx", adata_filter.X)
adata_filter.obs.to_csv("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_cell_metadata_filter.csv")
adata_filter.var.to_csv("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_gene_metadata_filter.csv")
mmwrite("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/monocle3/scRNA_adata_mono_csr_matrix_filter.mtx", csr_matrix(adata_filter.X))

# mTORC signaling score comparison

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")
adata

In [ ]:
adata.obs['monocluster'] = adata.obs['10_cluster'].replace({
    '0': 'mono0',
    '1': 'mono1',
    '2': 'mono2',
    '3': 'mono3',
    '4': 'mono4',
    '5': 'mono5',
    '6': 'mono6',
    '7': 'mono7',
    '8': 'mono8',
    '9': 'mono9'
})

sc.pl.umap(adata, color=['monocluster'])

In [ ]:
adata.obs['state'] = adata.obs['state'].replace({
    'healthy': 'Healthy',
    'NAFLD': 'MASL'
})

sc.pl.umap(adata, color=['state'])

In [ ]:
sc.pl.umap(adata, color=['TNF','IL1B','IL6','CD80','CD86','STAT1'], cmap='bwr')

In [ ]:
# GO:0006914 autophagy score
autophagy = {'ATG5','ATG7','ATG12','ATG14','ATG3','ATG4B','BECN1','MAP1LC3B','ULK1',
             'WIPI1','GABARAPL1','SQSTM1','RB1CC1','AMBRA1','ATG9A'}

sc.tl.score_genes(adata, gene_list=autophagy , score_name="autophagy_score")
sc.pl.umap(adata, color='autophagy_score', cmap='bwr', title='autophagy score')

In [ ]:
# Glycolysis score
glycolysis = {'HK2','HK3','GPI','PFKP','ALDOA','ALDOC','TPI1','GAPDH','PGK1','PGAM1','ENO1','ENO2','PKM','LDHA','SLC2A1','PDK1'}

sc.tl.score_genes(adata, gene_list=glycolysis , score_name="glycolysis_score")
sc.pl.umap(adata, color='glycolysis_score', cmap='bwr', title='glycolysis score')

In [ ]:
# Cytokine score
cytokine = {'IL1B','IL6','IL10','IL12B','IL18','TNF','CCL2','CCL3','CCL4','CCL5','CXCL8','CXCL10','IFNG','IL23A','IL1A','CSF1','CSF2','NFKB1'}

sc.tl.score_genes(adata, gene_list=cytokine , score_name="cytokine_score")
sc.pl.umap(adata, color='cytokine_score', cmap='bwr', title='cytokine score')

In [ ]:
# HALLMARK_MTORC_signaling 
mtorc = {'AKT1S1','ATF4','BCL2','CCND1','CDKN1A','DDIT4','EIF4EBP1','EIF4EBP2','EPRS',
'GADD45A','GSK3B','HIF1A','HSP90AA1','HSP90AB1','IRS1','LAMTOR1','LAMTOR2',
'LAMTOR3','LAMTOR4','LAMTOR5','MLST8','MTOR','RHEB','RPS6','RPS6KB1','RPTOR',
'SESN1','SLC7A5','TSC1','TSC2','ULK1','YWHAQ'}

sc.tl.score_genes(adata, gene_list=mtorc , score_name="mtorc_score")
sc.pl.umap(adata, color='mtorc_score', cmap='bwr', title='mtorc score')

In [ ]:
# LPS/TLR Response signaling - inflammation activity
lpstlr = {'TLR4','CD14','MYD88','TICAM1','LY96','NFKB1','NFKBIA','RELA',
'IRAK1','IRAK2','TRAF6','MAP3K7','MAPK8','MAPK14','IFNB1','TNF','IL1B','IL6','IL10',
'CCL2','CCL3','CCL4','CXCL8','CXCL10','STAT1','IRF3','IRF5','IRF7','SOCS3'}

sc.tl.score_genes(adata, gene_list=lpstlr , score_name="lpstlr_score")
sc.pl.umap(adata, color='lpstlr_score', cmap='bwr', title='lpstlr score')

In [ ]:
# M1 macrophage signature
m1 = {'IL1A', 'IL1B', 'IL6', 'TNF', 'NOS1',
'CD80','CD86','TLR2','TLR4','CD38','GPR18','FPR2',
'STAT1','IRF5'}

sc.tl.score_genes(adata, gene_list=m1 , score_name="m1_score")
sc.pl.umap(adata, color='m1_score', cmap='bwr', title='m1 score')

In [ ]:
# M2 macrophage signature
m2 = {'IL10', 'TGFB1', 'ARG1', 'CD163','CD206','CD301','MSR1',
'STAT3', 'STAT6', 'IRF4','PPARG','EGR2'}

sc.tl.score_genes(adata, gene_list=m1 , score_name="m2_score")
sc.pl.umap(adata, color='m2_score', cmap='bwr', title='m2 score')

In [ ]:
sig = ['autophagy_score','glycolysis_score','cytokine_score','mtorc_score','lpstlr_score']  # 예시 유전자 리스트
n_sig = len(sig)
fig, axs = plt.subplots(n_sig, 1, figsize=(6, 2 * n_sig))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_sig == 1:
    axs = [axs]

for i, sig in enumerate(sig):
    sc.pl.violin(
        adata,
        keys=sig,
        groupby='monocluster',
        ax=axs[i],
        #palette = palette,
        show=False,
        stripplot=True,
        jitter=True
    )
    axs[i].set_title(sig)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sig = ['autophagy_score','glycolysis_score','cytokine_score','mtorc_score','lpstlr_score']  # 예시 유전자 리스트
n_sig = len(sig)
fig, axs = plt.subplots(n_sig, 1, figsize=(2, 2 * n_sig))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_sig == 1:
    axs = [axs]

for i, sig in enumerate(sig):
    sc.pl.violin(
        adata,
        keys=sig,
        groupby='state',
        ax=axs[i],
        #palette = palette,
        show=False,
        stripplot=True,
        jitter=True
    )
    axs[i].set_title(sig)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

def fisher_test_score(adata, score, groupby="state"):
    df = adata.obs[[score, groupby]].dropna()

    # median 기준 high / low
    median = df[score].median()
    df["high"] = df[score] > median

    # contingency table
    table = pd.crosstab(df[groupby], df["high"])

    # MASL / Healthy 순서 보장
    table = table.loc[["MASL", "Healthy"]]

    _, p = fisher_exact(table)
    return p

In [ ]:
adata.obs['state'] = pd.Categorical(
    adata.obs['state'],
    categories=["Healthy", "MASL"],   # 👈 원하는 순서
    ordered=True
)

In [ ]:
sig_list = [
    'autophagy_score',
    'glycolysis_score',
    'cytokine_score',
    'mtorc_score',
    'lpstlr_score'
]

n_sig = len(sig_list)
fig, axs = plt.subplots(n_sig, 1, figsize=(2.5, 2 * n_sig))

if n_sig == 1:
    axs = [axs]

for i, sig in enumerate(sig_list):
    sc.pl.violin(
        adata,
        keys=sig,
        groupby='state',
        ax=axs[i],
        show=False,
        stripplot=True,
        jitter=True
    )

    # Fisher test
    p = fisher_test_score(adata, sig)
    star = p_to_star(p)

    # annotation 위치
    ymax = adata.obs[sig].max()
    axs[i].text(
        0.5, 0.8,
        star,
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        transform=axs[i].get_xaxis_transform()
    )

    axs[i].set_title(sig)

plt.tight_layout()
plt.show()

In [ ]:
sc.pl.dotplot(adata, ['glycolysis_score'
                     ], groupby='state')

In [ ]:
sc.pl.dotplot(adata, ['autophagy_score','glycolysis_score','cytokine_score','mtorc_score','lpstlr_score'], groupby='state')

In [ ]:
sc.pl.dotplot(adata, ['mtorc_score'], groupby='monocluster')

In [ ]:
res.res2d[res.res2d["Term"].str.contains("cyto", case=False)]

In [ ]:
import gseapy as gp

kegg = gp.get_library(name="KEGG_2021_Human")

mtor = kegg["mTOR signaling pathway"]
autophagy = kegg["Autophagy"]
glyco = kegg["Glycolysis / Gluconeogenesis"]
wnt = kegg["Wnt signaling pathway"]
pi3k = kegg["PI3K-Akt signaling pathway"]

In [ ]:
cytokine =kegg['Cytokine-cytokine receptor interaction']

In [ ]:
sc.tl.score_genes(adata, gene_list=mtor, score_name="mTOR_score")
sc.tl.score_genes(adata, gene_list=autophagy, score_name="autophagy_score")
sc.tl.score_genes(adata, gene_list=Glycolysis_Gluconeogenesis, score_name="glycolysis_score")
sc.tl.score_genes(adata, gene_list=wnt, score_name="wnt_score")
sc.tl.score_genes(adata, gene_list=pi3k, score_name="PI3K_score")

In [ ]:
sc.tl.score_genes(adata, gene_list=cytokine, score_name="Cytokine_score")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(
    data=adata.obs,
    x="monocluster",
    y="mTOR_score"
)
plt.show()

In [ ]:
sig = ['mTOR_score','autophagy_score','glycolysis_score','wnt_score','PI3K_score','Cytokine_score']  # 예시 유전자 리스트
n_sig = len(sig)
fig, axs = plt.subplots(n_sig, 1, figsize=(6, 2 * n_sig))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_sig == 1:
    axs = [axs]

for i, sig in enumerate(sig):
    sc.pl.violin(
        adata,
        keys=sig,
        groupby='monocluster',
        ax=axs[i],
        #palette = palette,
        show=False,
        stripplot=True,
        jitter=True
    )
    axs[i].set_title(sig)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sc.pl.dotplot(adata, ['mTOR_score','autophagy_score','glycolysis_score','wnt_score','PI3K_score'], groupby='state')

In [ ]:
sc.pl.dotplot(adata, ['Cytokine_score'], groupby='monocluster')

# pySCENIC analysis

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono03.h5ad")

In [ ]:
adata

In [ ]:
print(adata)  # AnnData object with n_obs × n_vars = ?
print(adata.X.__class__)  # numpy.ndarray or scipy.sparse.csr_matrix?

In [ ]:
print(adata.X.shape)

In [ ]:
#pre process adata to analyze with SCENIC
import numpy as np

print(adata.X.shape)

adata.layers = {}
adata.obsm = {}
adata.varm = {}
adata.obsp = {}

adata.obs = adata.obs[["n_counts"]] 
adata.var = adata.var[[]]

adata.var_names_make_unique()  
adata.var["Gene"] = adata.var_names
adata.obs["CellID"] = adata.obs.index
adata.write_loom("/home/jaehyunchoi/pySCENIC/data/scRNA_adata_mono03.loom", write_obsm_varm=False)

import loompy

with loompy.connect("/home/jaehyunchoi/pySCENIC/data/scRNA_adata_mono03.loom") as ds:
    print("Row attributes:", ds.ra.keys())  
    print("Column attributes:", ds.ca.keys())  

In [ ]:
adj_matrix = pd.read_csv('/home/jaehyunchoi/pySCENIC/data/cell_mono_SCENIC.adjacencies.tsv', index_col=0)

In [ ]:
adj_matrix = pd.read_csv('/home/jaehyunchoi/pySCENIC/data/cell_mono_SCENIC.adjacencies.tsv', index_col=0)
motif_matrix = pd.read_csv('/home/jaehyunchoi/pySCENIC/data/cell_mono_SCENIC.motifs.csv')
auc_matrix = pd.read_csv('/home/jaehyunchoi/pySCENIC/data/cell_mono_SCENIC.auc.csv', index_col=0)

# monocyte pySCENIC analysis

In [ ]:
adj_matrix = pd.read_csv('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.adjacencies.tsv', index_col=0)
motif_matrix = pd.read_csv('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.motifs.csv')
auc_matrix = pd.read_csv('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.auc.csv', index_col=0)

In [ ]:
motif_matrix

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono03.h5ad")
adata

In [ ]:
adata.obs = adata.obs.join(auc_matrix, how='left')
print(adata.obs)

In [ ]:
assert all(auc_matrix.index.isin(adata.obs.index)), "AUC index와 adata.obs index가 일치하지 않습니다."

In [ ]:
motif_matrix

In [ ]:
filtered_matrix = motif_matrix.iloc[:, [0, 8]]
filtered_matrix

In [ ]:
import ast 
filtered_matrix = motif_matrix.iloc[:, [0, 8]]
filtered_matrix = filtered_matrix.iloc[2:].reset_index(drop=True)
filtered_matrix.columns = ['TF', 'TargetGenes']
filtered_matrix['TargetGenes'] = filtered_matrix['TargetGenes'].apply(
 lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
filtered_matrix['TargetGenes'] = filtered_matrix['TargetGenes'].apply(
 lambda x: [gene[0] for gene in x])
regulons = filtered_matrix.groupby('TF')['TargetGenes'].apply(lambda x: sum(x, [])).to_dict()

In [ ]:
auc_data = adata.obs.loc[:, adata.obs.columns.str.endswith('(+)')]
clusters = adata.obs['monoMAC']
clustered_auc = auc_data.groupby(clusters).mean()
clustered_auc_1 = auc_data.groupby(clusters).mean()

In [ ]:
clustered_auc_1

# Mono3 analysis

In [ ]:
cluster_name = "mono3"
clusters = adata.obs['monoMAC']

adata.obs['mono3vs'] = clusters.apply(lambda x: x if x == 'mono3' else 'other')
cluster_auc = clustered_auc.loc[cluster_name]
top_20_tfs = cluster_auc.sort_values(ascending=False).head(20)
top_20_tfs.replace("(+)", "")
# 결과 출력
print(top_20_tfs)

In [ ]:
grouped_auc = auc_data.groupby(adata.obs['mono3vs']).mean()
grouped_auc

In [ ]:
from scipy.stats import ttest_ind

# 1번 클러스터와 other 그룹으로 데이터를 나누기
group_1 = auc_data[adata.obs['mono3vs'] == 'mono3']
group_other = auc_data[adata.obs['mono3vs'] == 'other']

# t-test 수행
p_values = {}
for tf in auc_data.columns:  # 각 TF에 대해 수행
    stat, p_val = ttest_ind(group_1[tf], group_other[tf], equal_var=False)
    p_values[tf] = p_val

# 결과를 정리하여 데이터프레임으로 변환
import pandas as pd
results = pd.DataFrame({
    'TF': p_values.keys(),
    'p-value': p_values.values()
}).set_index('TF')

# p-value로 정렬
results = results.sort_values('p-value')
print(results)

In [ ]:
from statsmodels.stats.multitest import multipletests
# 1. t-test로 p-value 계산
p_values = {}
z_scores = {}
for tf in auc_data.columns:
    # t-test
    stat, p_val = ttest_ind(group_1[tf], group_other[tf], equal_var=False)
    p_values[tf] = p_val
    
    # z-score
    mean_1, mean_other = group_1[tf].mean(), group_other[tf].mean()
    std_1, std_other = group_1[tf].std(), group_other[tf].std()
    z_scores[tf] = (mean_1 - mean_other) / np.sqrt((std_1**2 / len(group_1)) + (std_other**2 / len(group_other)))
    # p-values와 z-scores를 데이터프레임으로 정리
results = pd.DataFrame({
    'TF': auc_data.columns,
    'p-value': [p_values[tf] for tf in auc_data.columns],
    'z-score': [z_scores[tf] for tf in auc_data.columns]
}).set_index('TF')

# 2. FDR 보정
reject, pvals_corrected, _, _ = multipletests(results['p-value'], method='fdr_bh')
results['adjusted p-value'] = pvals_corrected
results['significant'] = reject  # True/False

# 3. -log10(adjusted p-value) 계산
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'])
results

In [ ]:
# z-score 기준으로 내림차순 정렬
results_sorted = results.sort_values(by='z-score', ascending=False)

# 유의미한 TF만 필터링 (adjusted p-value < 0.05)
sig_results = results_sorted[results_sorted['significant'] == True]

# 상위 20개만 보기
top_sig = sig_results.head(20)
top_sig

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(top_sig.index[::-1], top_sig['z-score'][::-1])
plt.xlabel('Z-score')
plt.title('Top 20 Significant TFs (mono3)')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Z-score 기준 상위 및 하위 6개 TF 선택
top_positive = results.nlargest(6, 'z-score')
top_negative = results.nsmallest(6, 'z-score')

# Volcano plot 생성
plt.figure(figsize=(10, 6))

# 유의미한 TF와 비유의미한 TF 분리
significant = results[results['significant']]
not_significant = results[~results['significant']]

# Volcano plot 그리기
plt.scatter(not_significant['z-score'], not_significant['-log10(adj p-value)'], color='grey', alpha=0.7, label='Not Significant')
plt.scatter(significant['z-score'], significant['-log10(adj p-value)'], color='red', alpha=0.7, label='Significant')

# z-score 기준 상위 및 하위 6개 TF 레이블 추가
for tf in top_positive.index:
    plt.text(
        top_positive.loc[tf, 'z-score'],
        top_positive.loc[tf, '-log10(adj p-value)'],
        tf,
        fontsize=8,
        color='blue'
    )

for tf in top_negative.index:
    plt.text(
        top_negative.loc[tf, 'z-score'],
        top_negative.loc[tf, '-log10(adj p-value)'],
        tf,
        fontsize=8,
        color='green'
    )

# 가이드라인 추가 (예: p-value 및 z-score 기준선)
plt.axhline(-np.log10(0.05), color='blue', linestyle='--', label='p = 0.05')
plt.axvline(-1, color='green', linestyle='--', label='z = -1')
plt.axvline(1, color='green', linestyle='--', label='z = 1')

# 축 레이블 및 제목
plt.xlabel('Z-score (Cluster 1 vs Other)', fontsize=12)
plt.ylabel('-log10(Adjusted p-value)', fontsize=12)
plt.title('Volcano Plot of TF AUC Scores', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

# Plot 표시
plt.show()

# 상위 및 하위 5개 TF 출력
print("Top 6 TFs with highest z-scores:")
print(top_positive)

print("\nTop 6 TFs with lowest z-scores:")
print(top_negative)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 무한 방지용 epsilon
epsilon = np.nextafter(0, 1)
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'] + epsilon)

# z-score 기준 정렬
results_sorted = results.sort_values(by='z-score', ascending=False)

# 상위 3개 TF
top5 = results_sorted.head(5)
print("Top 3 TFs in mono3 cluster:")
print(top5)

plt.figure(figsize=(10, 6))
bars = plt.bar(results_sorted.index[:20], results_sorted['z-score'][:20], alpha=0.6)

# 상위 3개 강조
for i, bar in enumerate(bars):
    tf_name = results_sorted.index[i]
    if tf_name in top5.index:
        bar.set_alpha(1.0)
        bar.set_edgecolor('black')
        bar.set_linewidth(1.5)

plt.xticks(rotation=75, ha='right')
plt.ylabel('Z-score')
plt.title('Top TFs in mono3 cluster (highlighted top 5)')
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 텍스트 라벨 추가
for tf in top5.index:
    y = results_sorted.loc[tf, 'z-score']
    plt.text(tf, y + 0.2, f"{tf}\n({y:.2f})", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
top5

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(results_sorted['z-score'], results_sorted['-log10(adj p-value)'],
            c='lightgray', alpha=0.6, label='All TFs')

# top3 강조
plt.scatter(top5['z-score'], top5['-log10(adj p-value)'],
            c='red', s=80, label='Top 5 TFs')

for tf in top5.index:
    x = top5.loc[tf, 'z-score']
    y = top5.loc[tf, '-log10(adj p-value)']
    plt.text(x, y + 0.3, tf, ha='center', fontsize=9, fontweight='bold', color='darkred')

plt.xlabel('Z-score')
plt.ylabel('-log10(FDR-adjusted p-value)')
plt.title('Significant TFs in mono3 cluster')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/Figure_script_IPF/Figures/SCENIC_volcanoplot_v2.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
results_sorted.head(10)

In [ ]:
clustered_auc_1

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
selected_tfs = ['FOSL1(+)', 'NFKB2(+)', 'RELB(+)','JUND(+)', 'NFKB1(+)', 'ZMIZ1(+)','ATF5(+)','TGIF1(+)','BHLHE40(+)','FOXO1(+)']  
subset_auc = clustered_auc[selected_tfs]
# Figure 사이즈 및 스타일 지정 (논문용)
plt.figure(figsize=(6, 4), dpi=300)  # 논문용 해상도와 크기
sns.set(font_scale=0.5)              # 글자 크기

# Heatmap
sns.heatmap(
    subset_auc, 
    cmap='bwr',   # 혹은 'coolwarm', 'magma', 'plasma' 등
    linewidths=0.5,   # 셀 간격
    linecolor='grey', # 셀 테두리 색
    annot=True,      # 값 표시 (원하면 True, 아니면 False)
    fmt=".2f",       # 소수점 두 자리
    cbar_kws={"label": "AUC Score"},  # 컬러바 라벨
    square=True      # 정사각형 셀
)

# 축 라벨 조정 (옵션)
plt.xlabel('Transcription Factors', fontsize=12)
plt.ylabel('Cluster', fontsize=12)

In [ ]:
motif_matrix.iloc[0, 0] = "TF"
motif_matrix.iloc[0, 1] = "MotifID"
selected_tfs = ['FOSL1', 'NFKB2', 'RELB','JUND', 'NFKB1', 'ZMIZ1','ATF5','TGIF1','BHLHE40','FOXO1'] # 'FOSL1(+)', 'NFKB2(+)', 'RELB(+)','JUND(+)', 'NFKB1(+)'
filtered_matrix = motif_matrix[motif_matrix['Unnamed: 0'].isin(selected_tfs)]
filtered_matrix

In [ ]:
filtered_matrix_01 = filtered_matrix[filtered_matrix['Enrichment.5'].str.contains('top5perTarget', na=False)]
filtered_matrix_01

In [ ]:
top5_5_matrix = filtered_matrix_01.iloc[:, [0,3,4,8]]
top5_5_matrix.columns = ['TF','NES','Qvalue', 'TargetGenes']
top5_5_matrix

In [ ]:
top5_5_matrix['TargetGenes'] = top5_5_matrix['TargetGenes'].astype(str).apply(eval)
top5_5_matrix_ex = top5_5_matrix.explode('TargetGenes')

top5_5_matrix_ex[['TargetGenes', 'importance']] = top5_5_matrix_ex['TargetGenes'].apply(
    lambda x: pd.Series({'TargetGenes': x[0], 'importance': x[1]})
)

In [ ]:
top5_5_matrix_ex.to_csv('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Top10_gene_mono3.motifs.csv')

In [ ]:
import networkx as nx
G = nx.from_pandas_edgelist(top5_5_matrix_ex, source='TF', target='TargetGenes', edge_attr='importance')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
pos = nx.spring_layout(G, k=0.3, iterations=50)

# TF와 Target 구분해서 색상 다르게
node_colors = ['red' if n in top5_5_matrix_ex['TF'].unique() else 'lightblue' for n in G.nodes()]

nx.draw(
    G, pos,
    with_labels=True,
    node_size=100,
    font_size=6,
    edge_color='gray',
    node_color=node_colors,
    alpha=0.8
)
plt.show()

# Mono1 analysis

In [ ]:
cluster_name = "mono1"
clusters = adata.obs['monoMAC']

adata.obs['mono1vs'] = clusters.apply(lambda x: x if x == 'mono1' else 'other')
cluster_auc = clustered_auc.loc[cluster_name]
top_20_tfs = cluster_auc.sort_values(ascending=False).head(20)
top_20_tfs.replace("(+)", "")
# 결과 출력
print(top_20_tfs)

In [ ]:
grouped_auc = auc_data.groupby(adata.obs['mono1vs']).mean()
grouped_auc

In [ ]:
from scipy.stats import ttest_ind

# 1번 클러스터와 other 그룹으로 데이터를 나누기
group_1 = auc_data[adata.obs['mono1vs'] == 'mono1']
group_other = auc_data[adata.obs['mono1vs'] == 'other']

# t-test 수행
p_values = {}
for tf in auc_data.columns:  # 각 TF에 대해 수행
    stat, p_val = ttest_ind(group_1[tf], group_other[tf], equal_var=False)
    p_values[tf] = p_val

# 결과를 정리하여 데이터프레임으로 변환
import pandas as pd
results = pd.DataFrame({
    'TF': p_values.keys(),
    'p-value': p_values.values()
}).set_index('TF')

# p-value로 정렬
results = results.sort_values('p-value')
print(results)

In [ ]:
from statsmodels.stats.multitest import multipletests
# 1. t-test로 p-value 계산
p_values = {}
z_scores = {}
for tf in auc_data.columns:
    # t-test
    stat, p_val = ttest_ind(group_1[tf], group_other[tf], equal_var=False)
    p_values[tf] = p_val
    
    # z-score
    mean_1, mean_other = group_1[tf].mean(), group_other[tf].mean()
    std_1, std_other = group_1[tf].std(), group_other[tf].std()
    z_scores[tf] = (mean_1 - mean_other) / np.sqrt((std_1**2 / len(group_1)) + (std_other**2 / len(group_other)))
    # p-values와 z-scores를 데이터프레임으로 정리
results = pd.DataFrame({
    'TF': auc_data.columns,
    'p-value': [p_values[tf] for tf in auc_data.columns],
    'z-score': [z_scores[tf] for tf in auc_data.columns]
}).set_index('TF')

# 2. FDR 보정
reject, pvals_corrected, _, _ = multipletests(results['p-value'], method='fdr_bh')
results['adjusted p-value'] = pvals_corrected
results['significant'] = reject  # True/False

# 3. -log10(adjusted p-value) 계산
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'])
results

In [ ]:
# z-score 기준으로 내림차순 정렬
results_sorted = results.sort_values(by='z-score', ascending=False)

# 유의미한 TF만 필터링 (adjusted p-value < 0.05)
sig_results = results_sorted[results_sorted['significant'] == True]

# 상위 20개만 보기
top_sig = sig_results.head(20)
top_sig

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(top_sig.index[::-1], top_sig['z-score'][::-1])
plt.xlabel('Z-score')
plt.title('Top 20 Significant TFs (mono1)')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Z-score 기준 상위 및 하위 6개 TF 선택
top_positive = results.nlargest(6, 'z-score')
top_negative = results.nsmallest(6, 'z-score')

# Volcano plot 생성
plt.figure(figsize=(10, 6))

# 유의미한 TF와 비유의미한 TF 분리
significant = results[results['significant']]
not_significant = results[~results['significant']]

# Volcano plot 그리기
plt.scatter(not_significant['z-score'], not_significant['-log10(adj p-value)'], color='grey', alpha=0.7, label='Not Significant')
plt.scatter(significant['z-score'], significant['-log10(adj p-value)'], color='red', alpha=0.7, label='Significant')

# z-score 기준 상위 및 하위 6개 TF 레이블 추가
for tf in top_positive.index:
    plt.text(
        top_positive.loc[tf, 'z-score'],
        top_positive.loc[tf, '-log10(adj p-value)'],
        tf,
        fontsize=8,
        color='blue'
    )

for tf in top_negative.index:
    plt.text(
        top_negative.loc[tf, 'z-score'],
        top_negative.loc[tf, '-log10(adj p-value)'],
        tf,
        fontsize=8,
        color='green'
    )

# 가이드라인 추가 (예: p-value 및 z-score 기준선)
plt.axhline(-np.log10(0.05), color='blue', linestyle='--', label='p = 0.05')
plt.axvline(-1, color='green', linestyle='--', label='z = -1')
plt.axvline(1, color='green', linestyle='--', label='z = 1')

# 축 레이블 및 제목
plt.xlabel('Z-score (Cluster 1 vs Other)', fontsize=12)
plt.ylabel('-log10(Adjusted p-value)', fontsize=12)
plt.title('Volcano Plot of TF AUC Scores', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

# Plot 표시
plt.show()

# 상위 및 하위 5개 TF 출력
print("Top 6 TFs with highest z-scores:")
print(top_positive)

print("\nTop 6 TFs with lowest z-scores:")
print(top_negative)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 무한 방지용 epsilon
epsilon = np.nextafter(0, 1)
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'] + epsilon)

# z-score 기준 정렬
results_sorted = results.sort_values(by='z-score', ascending=False)

# 상위 3개 TF
top5 = results_sorted.head(5)
print("Top 5 TFs in mono1 cluster:")
print(top5)

plt.figure(figsize=(10, 6))
bars = plt.bar(results_sorted.index[:20], results_sorted['z-score'][:20], alpha=0.6)

# 상위 3개 강조
for i, bar in enumerate(bars):
    tf_name = results_sorted.index[i]
    if tf_name in top5.index:
        bar.set_alpha(1.0)
        bar.set_edgecolor('black')
        bar.set_linewidth(1.5)

plt.xticks(rotation=75, ha='right')
plt.ylabel('Z-score')
plt.title('Top TFs in mono1 cluster (highlighted top 5)')
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 텍스트 라벨 추가
for tf in top5.index:
    y = results_sorted.loc[tf, 'z-score']
    plt.text(tf, y + 0.2, f"{tf}\n({y:.2f})", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(results_sorted['z-score'], results_sorted['-log10(adj p-value)'],
            c='lightgray', alpha=0.6, label='All TFs')

# top3 강조
plt.scatter(top5['z-score'], top5['-log10(adj p-value)'],
            c='red', s=80, label='Top 5 TFs')

for tf in top5.index:
    x = top5.loc[tf, 'z-score']
    y = top5.loc[tf, '-log10(adj p-value)']
    plt.text(x, y + 0.3, tf, ha='center', fontsize=9, fontweight='bold', color='darkred')

plt.xlabel('Z-score')
plt.ylabel('-log10(FDR-adjusted p-value)')
plt.title('Significant TFs in mono1 cluster')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/Figure_script_IPF/Figures/SCENIC_volcanoplot_v2.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
results_sorted.head(10)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
selected_tfs = ['FOSL2(+)', 'FOS(+)', 'NFE2(+)','SPI1(+)', 'ARID3A(+)', 'GABPB1(+)','YY1(+)','FOSB(+)','TGIF1(+)','NFE2L2(+)']  
subset_auc = clustered_auc[selected_tfs]
# Figure 사이즈 및 스타일 지정 (논문용)
plt.figure(figsize=(6, 4), dpi=300)  # 논문용 해상도와 크기
sns.set(font_scale=0.5)              # 글자 크기

# Heatmap
sns.heatmap(
    subset_auc, 
    cmap='bwr',   # 혹은 'coolwarm', 'magma', 'plasma' 등
    linewidths=0.5,   # 셀 간격
    linecolor='grey', # 셀 테두리 색
    annot=True,      # 값 표시 (원하면 True, 아니면 False)
    fmt=".2f",       # 소수점 두 자리
    cbar_kws={"label": "AUC Score"},  # 컬러바 라벨
    square=True      # 정사각형 셀
)

# 축 라벨 조정 (옵션)
plt.xlabel('Transcription Factors', fontsize=12)
plt.ylabel('Cluster', fontsize=12)

In [ ]:
motif_matrix.iloc[0, 0] = "TF"
motif_matrix.iloc[0, 1] = "MotifID"
selected_tfs = ['FOSL2', 'FOS', 'NFE2','SPI1', 'ARID3A', 'GABPB1','YY1','FOSB','TGIF1','NFE2L2']
filtered_matrix = motif_matrix[motif_matrix['Unnamed: 0'].isin(selected_tfs)]
filtered_matrix

In [ ]:
filtered_matrix_01 = filtered_matrix[filtered_matrix['Enrichment.5'].str.contains('top5perTarget', na=False)]
top5_5_matrix = filtered_matrix_01.iloc[:, [0,3,4,8]]
top5_5_matrix.columns = ['TF','NES','Qvalue', 'TargetGenes']
top5_5_matrix['TargetGenes'] = top5_5_matrix['TargetGenes'].astype(str).apply(eval)
top5_5_matrix_ex = top5_5_matrix.explode('TargetGenes')

top5_5_matrix_ex[['TargetGenes', 'importance']] = top5_5_matrix_ex['TargetGenes'].apply(
    lambda x: pd.Series({'TargetGenes': x[0], 'importance': x[1]})
)
#top5_5_matrix_ex.to_csv('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Top10_gene_mono1.motifs.csv')

In [ ]:
import networkx as nx
G = nx.from_pandas_edgelist(top5_5_matrix_ex, source='TF', target='TargetGenes', edge_attr='importance')
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
pos = nx.spring_layout(G, k=0.3, iterations=50)

# TF와 Target 구분해서 색상 다르게
node_colors = ['red' if n in top5_5_matrix_ex['TF'].unique() else 'lightblue' for n in G.nodes()]

nx.draw(
    G, pos,
    with_labels=True,
    node_size=100,
    font_size=6,
    edge_color='gray',
    node_color=node_colors,
    alpha=0.8
)
plt.show()